# LeetCode 335 — Self Crossing bằng Python (Hard)

Notebook này giảng nhập môn bài toán:

```python
class Solution:
    def isSelfCrossing(self, distance: List[int]) -> bool:

```

Tên bài dịch nôm na là **"Đường đi tự cắt chính nó"**. Ta đứng ở gốc tọa độ, lần lượt đi lên **Bắc**, sang **Tây**, xuống **Nam**, sang **Đông**, rồi lại lên Bắc… (cứ mỗi đoạn lại **rẽ trái**), độ dài từng đoạn cho trong mảng `distance`. Hỏi: đường đi có lúc nào **cắt** hoặc **chạm** vào phần đường đã đi trước đó không?

Đây là một bài Hard rất "lạ": code cuối cùng chỉ khoảng **10 dòng toàn phép so sánh `>=` và `<=`**, không có cấu trúc dữ liệu nào cả. Cái khó nằm ở **hình học**: phải tự thuyết phục mình rằng chỉ cần so đoạn mới với **3 đoạn rất gần** là đủ. Ta sẽ đi từng bước, từ cách người mới tự nghĩ ra cho tới cách tối ưu.

**Mục tiêu của bài học**

1. Biết đặt bài toán lên **hệ trục tọa độ**: điểm `(x, y)`, bốn hướng N/W/S/E, vector `(dx, dy)`, phép `i % 4` để xoay vòng hướng; tự vẽ đường đi bằng ASCII để *nhìn thấy* bài toán.
2. **Cách 1 — đi từng mét, rải "vụn bánh mì"**: lưu mọi điểm nguyên đã đi qua trong một `set`. Rất trực giác, đúng, nhưng chậm theo **tổng độ dài** (có thể tới `10^10` bước).
3. **Cách 2 — mô phỏng bằng đoạn thẳng**: lưu từng đoạn, kiểm tra đoạn mới có giao với mọi đoạn cũ không kề nhau không. Học cách kiểm tra **hai đoạn ngang/dọc giao nhau** (kể cả chạm) bằng **chồng khoảng** (interval overlap). Thời gian `O(n²)` — dùng làm **trọng tài** (đáp án chắc chắn đúng).
4. **Nhận xét then chốt** (rút ra bằng thí nghiệm rồi giải thích bằng hình): tại lần cắt đầu tiên, đoạn `i` luôn đụng một trong ba đoạn `i-3`, `i-4`, `i-5`. Có đúng **3 trường hợp hình học**, mỗi trường hợp có hình ASCII và điều kiện riêng.
5. **Cách 3 — lời giải chính**: duyệt một lượt, mỗi bước kiểm tra 3 trường hợp. Thời gian `O(n)`, bộ nhớ `O(1)`. Trace kỹ, nộp LeetCode, đối chiếu chéo với trọng tài trên dữ liệu ngẫu nhiên có `random.seed`.
6. **Cách 4 — "xoắn ra / xoắn vào"** (spiral out / spiral in): một góc nhìn khác, cũng `O(n)` thời gian, giúp hiểu *vì sao* chỉ cần nhìn vài đoạn gần.
7. Nhận diện các lỗi kinh điển (`>` thay `>=`, thiếu trường hợp, thiếu một điều kiện, **chỉ số âm** của Python, quên bước chuyển tiếp…), mẹo nhận dạng pattern và các bài anh em.

## 1. Đề bài

Đề LeetCode (dịch sát nghĩa):

> Cho một mảng số nguyên `distance`.
>
> Bạn bắt đầu tại điểm `(0, 0)` trên mặt phẳng X-Y. Bạn đi `distance[0]` mét về phía **Bắc**, sau đó `distance[1]` mét về phía **Tây**, `distance[2]` mét về phía **Nam**, `distance[3]` mét về phía **Đông**, và cứ thế tiếp tục. Nói cách khác, sau mỗi lần di chuyển, hướng đi của bạn đổi **ngược chiều kim đồng hồ** (counter-clockwise).
>
> Trả về `true` nếu đường đi của bạn **tự cắt** chính nó (crosses itself), ngược lại trả về `false`.

**Example 1**

```text
Input: distance = [2,1,1,2]
Output: true
Giải thích: Đường đi tự cắt tại điểm (0, 1).
```

**Example 2**

```text
Input: distance = [1,2,3,4]
Output: false
Giải thích: Đường đi không tự cắt tại bất kỳ điểm nào.
```

**Example 3**

```text
Input: distance = [1,1,1,2,1]
Output: true
Giải thích: Đường đi tự cắt tại điểm (0, 0).
```

**Constraints**

- `1 <= distance.length <= 10^5`
- `1 <= distance[i] <= 10^5`

### Ký hiệu dùng suốt notebook

- `n = len(distance)` — số đoạn đi.
- `d` — viết tắt cho `distance` trong công thức; `d[i]` là độ dài **đoạn thứ `i`** (đánh số từ `0`).
- "**Đoạn `i`**" = đoạn thẳng từ điểm cuối của đoạn `i-1` tới điểm cuối của chính nó.
- `S = d[0] + d[1] + … + d[n-1]` — **tổng độ dài** đường đi (tối đa `10^5 · 10^5 = 10^10`).

### 1.1 Hiểu đề bằng tay với Example 1

`distance = [2, 1, 1, 2]`. Trục `y` hướng lên trên là **Bắc**, trục `x` hướng sang phải là **Đông**.

| Đoạn | Hướng | Độ dài | Từ điểm | Tới điểm | Ghi chú |
|---:|---|---:|---|---|---|
| 0 | Bắc (N): `y` tăng | 2 | `(0, 0)` | `(0, 2)` | đi thẳng lên |
| 1 | Tây (W): `x` giảm | 1 | `(0, 2)` | `(-1, 2)` | rẽ trái |
| 2 | Nam (S): `y` giảm | 1 | `(-1, 2)` | `(-1, 1)` | rẽ trái |
| 3 | Đông (E): `x` tăng | 2 | `(-1, 1)` | `(1, 1)` | đi qua `(0, 1)` — điểm này nằm trên đoạn 0! |

Đoạn 3 nằm trên đường `y = 1`, chạy từ `x = -1` tới `x = 1`, nên nó đi **xuyên qua** điểm `(0, 1)`. Mà đoạn 0 là đoạn dọc `x = 0` từ `y = 0` tới `y = 2`, cũng chứa `(0, 1)`. Hai đoạn cắt nhau → đáp án `true`.

Cell dưới làm đúng các bước trên bằng 4 phép cộng/trừ, chưa cần hàm hay vòng lặp.

In [1]:
distance = [2, 1, 1, 2]
x, y = 0, 0
print("Xuất phát tại      ", (x, y))

y += distance[0]   # Bắc: y tăng
print("Sau đoạn 0 (Bắc 2):", (x, y))
x -= distance[1]   # Tây: x giảm
print("Sau đoạn 1 (Tây 1):", (x, y))
y -= distance[2]   # Nam: y giảm
print("Sau đoạn 2 (Nam 1):", (x, y))
x += distance[3]   # Đông: x tăng
print("Sau đoạn 3 (Đông 2):", (x, y))

Xuất phát tại       (0, 0)
Sau đoạn 0 (Bắc 2): (0, 2)
Sau đoạn 1 (Tây 1): (-1, 2)
Sau đoạn 2 (Nam 1): (-1, 1)
Sau đoạn 3 (Đông 2): (1, 1)


### 1.2 "Tự cắt" nghĩa là gì? (đọc kỹ, đây là chỗ hay sai)

Đề dùng chữ *crosses*, nhưng nhìn Example 3 ta thấy nghĩa thực tế rộng hơn "cắt hình chữ thập":

- **Cắt xuyên qua** (hình dấu `+`): như Example 1, đoạn 3 xuyên qua đoạn 0.
- **Chạm vào** (hình chữ `T`): đầu mút của đoạn mới **chạm** vào thân hoặc đầu của đoạn cũ cũng tính là cắt. Ví dụ `[1, 1, 1, 1]`: đi một vòng vuông rồi **quay về đúng** `(0, 0)` → `true`.
- **Đè lên nhau** (cùng nằm trên một đường thẳng và có phần chung): cũng tính là cắt. Ví dụ `[1, 1, 2, 1, 1]`: đoạn 4 đi lên dọc theo đường `x = 0` và chạm đầu dưới của đoạn 0 tại `(0, 0)`.

Nói gọn: **hai đoạn có ít nhất một điểm chung** là cắt. Ngoại lệ duy nhất: hai đoạn **liền kề** (đoạn `i` và `i+1`) đương nhiên chung điểm rẽ, cái đó **không** tính.

Example 3 `[1, 1, 1, 2, 1]`: các điểm là `(0,0) → (0,1) → (-1,1) → (-1,0) → (1,0) → (1,1)`. Đoạn 3 đi trên `y = 0` từ `x = -1` tới `x = 1`, đi qua `(0, 0)` — chính là điểm xuất phát (đầu dưới của đoạn 0). Đó là lý do đề ghi "tự cắt tại điểm (0, 0)".

> Ghi nhớ: trong bài này, **chạm cũng là cắt**. Vì thế mọi so sánh trong lời giải sẽ là `>=` / `<=`, không phải `>` / `<`.

## 2. Nền tảng

### 2.1 Bốn hướng và phép `i % 4`

Mỗi hướng đi có thể viết thành một **vector** `(dx, dy)` = "đi 1 mét theo hướng này thì `x` đổi bao nhiêu, `y` đổi bao nhiêu":

| Hướng | Tiếng Anh | `(dx, dy)` | Đoạn có chỉ số |
|---|---|---|---|
| Bắc | North (N) | `( 0,  1)` | `0, 4, 8, …` |
| Tây | West (W) | `(-1,  0)` | `1, 5, 9, …` |
| Nam | South (S) | `( 0, -1)` | `2, 6, 10, …` |
| Đông | East (E) | `( 1,  0)` | `3, 7, 11, …` |

Hướng **lặp lại sau mỗi 4 đoạn**, nên hướng của đoạn `i` chỉ phụ thuộc vào **phần dư** `i % 4` (phép `%` = chia lấy dư: `5 % 4 = 1`, `8 % 4 = 0`). Ta lưu 4 vector vào một list `DIRS` theo đúng thứ tự N, W, S, E; khi đó hướng của đoạn `i` là `DIRS[i % 4]`.

Đi `L` mét theo vector `(dx, dy)` từ điểm `(x, y)` thì tới điểm `(x + dx·L, y + dy·L)`. Ví dụ đứng ở `(0, 2)`, đi Tây (`(-1, 0)`) 3 mét → `(0 - 3, 2 + 0) = (-3, 2)`.

Hai quan sát nhỏ nhưng dùng mãi về sau:

- Đoạn `i` và đoạn `i+2` luôn **song song, ngược chiều** (N với S, W với E).
- Đoạn `i` và đoạn `i±1`, `i±3`, `i±5`… luôn **vuông góc**; đoạn `i` và `i±2`, `i±4`, `i±6`… luôn **song song**. Tóm lại: hai đoạn cách nhau một số **lẻ** thì vuông góc, cách nhau số **chẵn** thì song song.

In [2]:
DIRS = [(0, 1), (-1, 0), (0, -1), (1, 0)]        # Bắc, Tây, Nam, Đông
DIR_NAMES = ["Bắc (N)", "Tây (W)", "Nam (S)", "Đông (E)"]

for i in range(9):
    dx, dy = DIRS[i % 4]
    print(f"đoạn {i}: i % 4 = {i % 4} -> {DIR_NAMES[i % 4]:9s} (dx, dy) = ({dx:2d}, {dy:2d})")

đoạn 0: i % 4 = 0 -> Bắc (N)   (dx, dy) = ( 0,  1)
đoạn 1: i % 4 = 1 -> Tây (W)   (dx, dy) = (-1,  0)
đoạn 2: i % 4 = 2 -> Nam (S)   (dx, dy) = ( 0, -1)
đoạn 3: i % 4 = 3 -> Đông (E)  (dx, dy) = ( 1,  0)
đoạn 4: i % 4 = 0 -> Bắc (N)   (dx, dy) = ( 0,  1)
đoạn 5: i % 4 = 1 -> Tây (W)   (dx, dy) = (-1,  0)
đoạn 6: i % 4 = 2 -> Nam (S)   (dx, dy) = ( 0, -1)
đoạn 7: i % 4 = 3 -> Đông (E)  (dx, dy) = ( 1,  0)
đoạn 8: i % 4 = 0 -> Bắc (N)   (dx, dy) = ( 0,  1)


### 2.2 Tuple, `enumerate` và danh sách các điểm

Một điểm được lưu bằng **tuple** `(x, y)` — giống list nhưng **không sửa được** (immutable), nhờ vậy có thể bỏ vào `set` hoặc làm key của `dict` (list thì không được). Viết `x, y = point` là **mở gói** (unpack) tuple thành hai biến.

`enumerate(distance)` cho ta **cùng lúc** chỉ số `i` và giá trị `length`:

```python
for i, length in enumerate([2, 1, 1, 2]):
    # (i, length) lần lượt là (0, 2), (1, 1), (2, 1), (3, 2)
```

Ta cần chỉ số `i` để biết hướng (`DIRS[i % 4]`) và cần `length` để biết đi bao xa. Hàm `path_points` dưới đây trả về **list `n + 1` điểm**: điểm xuất phát `(0, 0)` và điểm cuối của từng đoạn. Đoạn `i` chính là đoạn nối `points[i]` với `points[i + 1]`.

In [3]:
def path_points(distance):
    # Điểm xuất phát + điểm cuối của từng đoạn (tổng cộng n + 1 điểm).
    x, y = 0, 0
    points = [(x, y)]
    for i, length in enumerate(distance):
        dx, dy = DIRS[i % 4]
        x += dx * length
        y += dy * length
        points.append((x, y))
    return points


for d in ([2, 1, 1, 2], [1, 2, 3, 4], [1, 1, 1, 2, 1]):
    print(d, "->", path_points(d))

[2, 1, 1, 2] -> [(0, 0), (0, 2), (-1, 2), (-1, 1), (1, 1)]
[1, 2, 3, 4] -> [(0, 0), (0, 1), (-2, 1), (-2, -2), (2, -2)]
[1, 1, 1, 2, 1] -> [(0, 0), (0, 1), (-1, 1), (-1, 0), (1, 0), (1, 1)]


### 2.3 Một công cụ vẽ: nhìn thấy đường đi bằng ASCII

Bài này là bài **hình học**, nhìn hình sẽ hiểu nhanh gấp mười lần nhìn số. Hàm `draw_path` dưới đây vẽ đường đi ra màn hình bằng ký tự (bạn **không cần** hiểu từng dòng của nó, chỉ cần biết cách đọc hình):

| Ký hiệu | Ý nghĩa |
|---|---|
| `0`, `1`, `2`, … | ô thuộc đoạn có chỉ số đó (chỉ in chữ số cuối: đoạn 12 in là `2`) |
| `+` | góc rẽ: chỗ hai đoạn **liền kề** gặp nhau |
| `X` | chỗ hai đoạn **không liền kề** có điểm chung — tức là **đường đi tự cắt** |
| `S` | điểm xuất phát `(0, 0)` (nếu bị cắt ngay tại đó thì in `X`) |
| `*` | điểm kết thúc |

Mỗi mét theo chiều ngang được vẽ rộng 3 ký tự, theo chiều dọc cao 2 dòng (vì ký tự chữ cao hơn rộng), để hình đỡ méo. Bắc ở trên, Đông ở bên phải.

In [4]:
def draw_path(distance, sx=3, sy=2):
    # Vẽ đường đi bằng ký tự. sx, sy = số ký tự cho 1 mét theo chiều ngang / dọc.
    pts = path_points(distance)
    xs = [p[0] for p in pts]
    ys = [p[1] for p in pts]
    minx, maxx, miny, maxy = min(xs), max(xs), min(ys), max(ys)
    width, height = (maxx - minx) * sx + 1, (maxy - miny) * sy + 1
    owner = {}                                   # ô (hàng, cột) -> tập các đoạn đi qua ô đó
    for i in range(len(distance)):
        (x1, y1), (x2, y2) = pts[i], pts[i + 1]
        c1, r1 = (x1 - minx) * sx, (maxy - y1) * sy
        c2, r2 = (x2 - minx) * sx, (maxy - y2) * sy
        steps = max(abs(c2 - c1), abs(r2 - r1))
        for s in range(steps + 1):
            cell = (r1 + (r2 - r1) * s // steps, c1 + (c2 - c1) * s // steps)
            owner.setdefault(cell, set()).add(i)
    grid = [[" "] * width for _ in range(height)]
    for (r, c), segs in owner.items():
        if any(abs(a - b) >= 2 for a in segs for b in segs):
            grid[r][c] = "X"                     # hai đoạn không liền kề gặp nhau
        elif len(segs) == 2:
            grid[r][c] = "+"                     # góc rẽ
        else:
            grid[r][c] = str(next(iter(segs)) % 10)
    for (px, py), mark in ((pts[-1], "*"), (pts[0], "S")):
        r, c = (maxy - py) * sy, (px - minx) * sx
        if grid[r][c] != "X":
            grid[r][c] = mark
    print("\n".join("".join(row).rstrip() for row in grid))


for d in ([2, 1, 1, 2], [1, 2, 3, 4], [1, 1, 1, 2, 1]):
    print("distance =", d)
    draw_path(d)
    print()

distance = [2, 1, 1, 2]
+11+
2  0
+33X33*
   0
   S

distance = [1, 2, 3, 4]
+11111+
2     0
2     S
2
2
2
+33333333333*

distance = [1, 1, 1, 2, 1]
+11+  *
2  0  4
+33X33+



### 2.4 Đọc hình

- **Example 1** `[2, 1, 1, 2]`: đoạn `0` đi lên từ `S`, đoạn `1` sang trái, đoạn `2` xuống, đoạn `3` sang phải và **xuyên qua** đoạn `0` tại chữ `X` — đúng điểm `(0, 1)`.
- **Example 2** `[1, 2, 3, 4]`: mỗi đoạn dài hơn đoạn song song trước nó, đường đi **toả ra ngoài** như vỏ ốc, không bao giờ quay lại vùng cũ → `false`.
- **Example 3** `[1, 1, 1, 2, 1]`: đoạn `3` chạy ngang qua đúng điểm xuất phát, nên chữ `S` bị thay bằng `X` → `true`.

Từ giờ, mỗi khi gặp một input lạ, hãy gọi `draw_path` để nhìn. Đó là thói quen quan trọng nhất của bài này.

## 3. Cách 1 — Đi từng mét, rải "vụn bánh mì"

### 3.1 Ý tưởng tự nhiên nhất

Hãy tưởng tượng bạn đi trong rừng và cứ mỗi bước lại thả một mẩu bánh mì. Nếu có lúc bạn bước lên một chỗ **đã có bánh mì**, nghĩa là bạn đã quay lại đường cũ → đường đi tự cắt.

Trên máy tính, "bánh mì" là một **`set`** chứa các điểm đã đi qua. `set` (tập hợp) trong Python:

- lưu các phần tử **không trùng nhau**;
- `visited.add(p)` thêm phần tử `p`;
- `p in visited` kiểm tra `p` đã có chưa — nhanh, trung bình `O(1)` (nhờ băm / hashing), không phải dò lần lượt như list.

Phần tử của `set` phải là kiểu **không sửa được** — tuple `(x, y)` thì được, list `[x, y]` thì không.

In [5]:
visited = {(0, 0)}          # set có sẵn điểm xuất phát
visited.add((0, 1))
visited.add((0, 2))
visited.add((0, 1))         # thêm trùng: set không đổi
print(visited, "| số phần tử:", len(visited))
print("(0, 1) đã đi qua chưa?", (0, 1) in visited)
print("(5, 5) đã đi qua chưa?", (5, 5) in visited)

{(0, 1), (0, 2), (0, 0)} | số phần tử: 3
(0, 1) đã đi qua chưa? True
(5, 5) đã đi qua chưa? False


### 3.2 Vì sao chỉ cần lưu các điểm **nguyên**?

Đường đi thật là các đoạn thẳng liền, có vô số điểm. Nhưng ta chỉ cần lưu các **điểm có tọa độ nguyên** (lattice point), vì:

- Mọi độ dài `d[i]` đều là số nguyên, xuất phát từ `(0, 0)`, nên **mọi góc rẽ** đều có tọa độ nguyên, và mọi đoạn đều nằm trên một đường thẳng `x = số nguyên` (đoạn dọc) hoặc `y = số nguyên` (đoạn ngang).
- Một đoạn ngang `y = a` cắt một đoạn dọc `x = b` thì giao điểm là `(b, a)` — **nguyên**.
- Hai đoạn cùng nằm trên một đường thẳng mà có phần chung, thì phần chung là một đoạn có hai đầu là đầu mút nguyên của chúng — cũng chứa điểm nguyên.

Vậy: **đường đi tự cắt ⇔ có một điểm nguyên bị đi qua hai lần**. Ta đi từng mét (mỗi bước cộng `(dx, dy)` một lần), sau mỗi bước kiểm tra điểm mới đã có trong `visited` chưa.

Chú ý hai chi tiết:

1. Phải bỏ `(0, 0)` vào `visited` **ngay từ đầu** — nếu không, Example 3 (cắt tại điểm xuất phát) sẽ bị bỏ sót.
2. Điểm góc rẽ chỉ được thêm **một lần** (lúc kết thúc đoạn trước); đoạn sau bắt đầu bằng bước *tiếp theo*, nên hai đoạn liền kề không bị báo nhầm là cắt.

In [6]:
def is_self_crossing_grid(distance):
    x, y = 0, 0
    visited = {(0, 0)}
    for i, length in enumerate(distance):
        dx, dy = DIRS[i % 4]
        for _ in range(length):          # đi từng mét
            x += dx
            y += dy
            if (x, y) in visited:        # dẫm lên vụn bánh mì cũ
                return True
            visited.add((x, y))
    return False


for d in ([2, 1, 1, 2], [1, 2, 3, 4], [1, 1, 1, 2, 1], [1, 1, 1, 1], [1, 1, 2, 1, 1]):
    print(d, "->", is_self_crossing_grid(d))

[2, 1, 1, 2] -> True
[1, 2, 3, 4] -> False
[1, 1, 1, 2, 1] -> True
[1, 1, 1, 1] -> True
[1, 1, 2, 1, 1] -> True


### 3.3 Trace Cách 1 với Example 1

`distance = [2, 1, 1, 2]`. Mỗi dòng là **một bước 1 mét**:

| Bước | Đoạn | Hướng | Điểm mới | Đã có trong `visited`? | Hành động |
|---:|---:|---|---|---|---|
| 1 | 0 | Bắc | `(0, 1)` | không | thêm |
| 2 | 0 | Bắc | `(0, 2)` | không | thêm |
| 3 | 1 | Tây | `(-1, 2)` | không | thêm |
| 4 | 2 | Nam | `(-1, 1)` | không | thêm |
| 5 | 3 | Đông | `(0, 1)` | **có** (từ bước 1) | trả `True` |

Hàm `trace_grid` in đúng bảng này.

In [7]:
def trace_grid(distance):
    x, y = 0, 0
    visited = {(0, 0)}
    step = 0
    for i, length in enumerate(distance):
        dx, dy = DIRS[i % 4]
        for _ in range(length):
            x += dx
            y += dy
            step += 1
            seen = (x, y) in visited
            print(f"bước {step}: đoạn {i} ({DIR_NAMES[i % 4]}) -> điểm {(x, y)}, đã có? {seen}")
            if seen:
                print("=> dẫm lại điểm cũ, trả về True")
                return True
            visited.add((x, y))
    print("=> đi hết mà không dẫm lại điểm nào, trả về False")
    return False


trace_grid([2, 1, 1, 2])

bước 1: đoạn 0 (Bắc (N)) -> điểm (0, 1), đã có? False
bước 2: đoạn 0 (Bắc (N)) -> điểm (0, 2), đã có? False
bước 3: đoạn 1 (Tây (W)) -> điểm (-1, 2), đã có? False
bước 4: đoạn 2 (Nam (S)) -> điểm (-1, 1), đã có? False
bước 5: đoạn 3 (Đông (E)) -> điểm (0, 1), đã có? True
=> dẫm lại điểm cũ, trả về True


True

### 3.4 Cách 1 vướng ở đâu?

Số bước của Cách 1 bằng **tổng độ dài** `S = d[0] + … + d[n-1]`, và `set` cũng có thể chứa tới `S` điểm:

- Thời gian: `O(S)` (trung bình, do `set`).
- Bộ nhớ: `O(S)`.

Với constraints `n ≤ 10^5`, `d[i] ≤ 10^5`, trường hợp xấu nhất là `S = 10^10` — mười tỉ bước, và mười tỉ tuple trong bộ nhớ (hàng trăm GB). Không thể chạy được.

Cell dưới đo thử thời gian khi `S ≈ 10^6` (một đường **xoắn ra** không bao giờ cắt, nên phải đi hết) rồi ước lượng cho `S = 10^10`.

In [8]:
import time

big = [1000 * k for k in range(1, 45)]          # xoắn ra, không bao giờ cắt
S = sum(big)
t0 = time.perf_counter()
ans = is_self_crossing_grid(big)
elapsed = time.perf_counter() - t0
print(f"S = {S:,} bước, kết quả = {ans}, mất {elapsed:.2f} giây")
print(f"ước lượng cho S = 10^10: khoảng {elapsed * 1e10 / S / 3600:.0f} giờ (chưa kể hết RAM)")

S = 990,000 bước, kết quả = False, mất 0.60 giây
ước lượng cho S = 10^10: khoảng 2 giờ (chưa kể hết RAM)


## 4. Cách 2 — Mô phỏng bằng đoạn thẳng (trọng tài `O(n²)`)

### 4.1 Từ "từng điểm" sang "từng đoạn"

Cách 1 chậm vì nó làm việc với từng **mét**. Nhưng đường đi chỉ có `n` **đoạn** (tối đa `10^5`), dù mỗi đoạn dài bao nhiêu. Vậy ta lưu từng đoạn dưới dạng cặp điểm `(điểm đầu, điểm cuối)` và tự hỏi: *đoạn mới có giao với đoạn cũ nào không?*

Mọi đoạn trong bài đều **song song với trục tọa độ**: đoạn dọc (Bắc/Nam, `x` không đổi) hoặc đoạn ngang (Tây/Đông, `y` không đổi). Nhờ vậy kiểm tra giao nhau rất đơn giản, không cần công thức hình học phức tạp.

In [9]:
def segments_of(distance):
    # List n đoạn, mỗi đoạn là (điểm đầu, điểm cuối).
    pts = path_points(distance)
    return [(pts[i], pts[i + 1]) for i in range(len(distance))]


for i, seg in enumerate(segments_of([2, 1, 1, 2])):
    kind = "dọc " if seg[0][0] == seg[1][0] else "ngang"
    print(f"đoạn {i} ({DIR_NAMES[i % 4]:9s}, {kind}): {seg[0]} -> {seg[1]}")

đoạn 0 (Bắc (N)  , dọc ): (0, 0) -> (0, 2)
đoạn 1 (Tây (W)  , ngang): (0, 2) -> (-1, 2)
đoạn 2 (Nam (S)  , dọc ): (-1, 2) -> (-1, 1)
đoạn 3 (Đông (E) , ngang): (-1, 1) -> (1, 1)


### 4.2 Viên gạch đầu tiên: hai **khoảng** có chồng lên nhau không?

Trước khi nói về đoạn thẳng trên mặt phẳng, hãy giải bài 1 chiều: trên trục số, khoảng `[a, b]` và khoảng `[c, d]` có điểm chung không?

Phần chung (nếu có) bắt đầu ở **đầu trái lớn hơn** và kết thúc ở **đầu phải nhỏ hơn**:

```text
lo = max(a, c)      hi = min(b, d)      có điểm chung  ⇔  lo <= hi
```

Ví dụ:

| Khoảng 1 | Khoảng 2 | `lo` | `hi` | Chồng nhau? |
|---|---|---:|---:|---|
| `[1, 5]` | `[3, 8]` | 3 | 5 | có, phần chung `[3, 5]` |
| `[1, 2]` | `[3, 4]` | 3 | 2 | không (`3 > 2`) |
| `[1, 3]` | `[3, 6]` | 3 | 3 | **có** — chỉ chạm tại `3` (vẫn tính!) |
| `[4, 4]` | `[1, 6]` | 4 | 4 | có — "khoảng" một điểm nằm trong khoảng kia |

Dấu `<=` (chứ không phải `<`) chính là thứ giúp ta bắt được trường hợp **chạm**.

Một chi tiết: đoạn đi về phía Tây hoặc Nam có tọa độ đầu **lớn hơn** tọa độ cuối (ví dụ `x` đi từ `0` xuống `-1`). Vì thế hàm `overlap` nhận hai đầu theo thứ tự bất kỳ và tự lấy `min`/`max`.

In [10]:
def overlap(a, b, c, d):
    # Khoảng giữa a, b và khoảng giữa c, d (thứ tự đầu mút tùy ý) có điểm chung không?
    lo = max(min(a, b), min(c, d))
    hi = min(max(a, b), max(c, d))
    return lo <= hi


print(overlap(1, 5, 3, 8))    # True
print(overlap(1, 2, 3, 4))    # False
print(overlap(1, 3, 3, 6))    # True: chạm tại 3
print(overlap(4, 4, 1, 6))    # True: một điểm nằm trong khoảng
print(overlap(5, 1, 8, 3))    # True: đầu mút viết ngược thứ tự vẫn đúng

True
False
True
True
True


### 4.3 Hai đoạn ngang/dọc giao nhau ⇔ chồng theo `x` **và** chồng theo `y`

Với hai đoạn song song trục, ta chỉ cần kiểm tra **cả hai** điều:

1. khoảng `x` của chúng chồng nhau, **và**
2. khoảng `y` của chúng chồng nhau.

Tại sao đủ? Xét các tình huống:

| Tình huống | Ví dụ | Giải thích |
|---|---|---|
| Ngang × dọc, cắt chữ thập | `(-1,1)→(1,1)` và `(0,0)→(0,2)` | `x`: `[-1,1]` chứa `0`; `y`: `[0,2]` chứa `1` → chung `(0, 1)` |
| Ngang × dọc, chạm chữ T | `(-1,0)→(0,0)` và `(0,0)→(0,2)` | chung đúng một điểm `(0, 0)` |
| Hai đoạn ngang **cùng** đường thẳng | `(0,0)→(3,0)` và `(2,0)→(5,0)` | `y` đều là `[0,0]`, `x` chồng `[2,3]` → đè lên nhau |
| Hai đoạn ngang **khác** đường thẳng | `(0,0)→(3,0)` và `(0,1)→(3,1)` | `y`: `[0,0]` và `[1,1]` không chồng → không giao |
| Ngang × dọc, lệch nhau | `(-1,1)→(1,1)` và `(5,0)→(5,2)` | `x`: `[-1,1]` không chứa `5` → không giao |

Đây cũng chính là phép kiểm tra **hai hình chữ nhật chồng nhau** (bounding box overlap) — một đoạn ngang/dọc chỉ là một "hình chữ nhật dẹt". Với đoạn song song trục, phép kiểm tra này **chính xác tuyệt đối**, không có trường hợp ngoại lệ.

In [11]:
def segments_intersect(s, t):
    # Hai đoạn song song trục tọa độ có ít nhất một điểm chung không (kể cả chạm)?
    (x1, y1), (x2, y2) = s
    (x3, y3), (x4, y4) = t
    return overlap(x1, x2, x3, x4) and overlap(y1, y2, y3, y4)


tests = [
    (((-1, 1), (1, 1)), ((0, 0), (0, 2)), True),    # chữ thập
    (((-1, 0), (0, 0)), ((0, 0), (0, 2)), True),    # chữ T / chạm đầu
    (((0, 0), (3, 0)), ((2, 0), (5, 0)), True),     # cùng đường thẳng, đè nhau
    (((0, 0), (3, 0)), ((0, 1), (3, 1)), False),    # song song khác đường
    (((-1, 1), (1, 1)), ((5, 0), (5, 2)), False),   # lệch nhau
]
for s, t, expected in tests:
    got = segments_intersect(s, t)
    print(f"{s} vs {t}: {got}")
    assert got == expected

((-1, 1), (1, 1)) vs ((0, 0), (0, 2)): True
((-1, 0), (0, 0)) vs ((0, 0), (0, 2)): True
((0, 0), (3, 0)) vs ((2, 0), (5, 0)): True
((0, 0), (3, 0)) vs ((0, 1), (3, 1)): False
((-1, 1), (1, 1)) vs ((5, 0), (5, 2)): False


### 4.4 Những đoạn nào **không** cần (và không được) so sánh?

Khi xét đoạn `i`, ta so nó với các đoạn cũ `j < i`, nhưng phải loại hai đoạn:

- **`j = i - 1` (liền kề)**: luôn chung điểm rẽ. Nếu so, `segments_intersect` sẽ luôn trả `True` → báo cắt sai với mọi input có từ 2 đoạn trở lên.
- **`j = i - 2`**: đoạn `i-2` song song **ngược chiều** với đoạn `i`, và hai đoạn này bị đoạn `i-1` (dài ít nhất `1` mét) đẩy ra **hai đường thẳng khác nhau**. Ví dụ đoạn 0 nằm trên `x = 0`, đoạn 2 nằm trên `x = -d[1] ≤ -1`. Nên chúng **không bao giờ** giao nhau — so cũng chỉ phí công.

Vậy chỉ cần `j = 0, 1, …, i - 3`, tức là `for j in range(i - 2)`. Cell dưới kiểm chứng hai khẳng định trên với 500 đường đi ngẫu nhiên.

In [12]:
import random

random.seed(2)
adj_always, two_never = True, True
for _ in range(500):
    d = [random.randint(1, 6) for _ in range(random.randint(3, 12))]
    segs = segments_of(d)
    for i in range(1, len(segs)):
        adj_always &= segments_intersect(segs[i], segs[i - 1])
        if i >= 2:
            two_never &= not segments_intersect(segs[i], segs[i - 2])
print("Đoạn i và i-1 LUÔN chung điểm:     ", adj_always)
print("Đoạn i và i-2 KHÔNG BAO GIỜ giao nhau:", two_never)

Đoạn i và i-1 LUÔN chung điểm:      True
Đoạn i và i-2 KHÔNG BAO GIỜ giao nhau: True


### 4.5 Thuật toán mô phỏng

```text
segs = danh sách n đoạn
với mỗi i = 0, 1, …, n-1:
    với mỗi j = 0, 1, …, i-3:
        nếu segs[i] giao segs[j]: trả True
trả False
```

Thứ tự duyệt `i` tăng dần nghĩa là ta "vẽ" từng đoạn và kiểm tra ngay; lần đầu thấy giao là dừng. Ta viết thêm `first_crossing(distance)`: trả về `(i, [các j bị đoạn i chạm])` cho **đoạn đầu tiên** gây cắt — hàm này rất hữu ích để làm thí nghiệm ở phần sau.

Đây là lời giải **chắc chắn đúng** (nó làm đúng định nghĩa "có hai đoạn không liền kề chung điểm"), nên từ giờ ta gọi nó là **trọng tài**: mọi cách nhanh hơn đều phải cho cùng kết quả với nó.

In [13]:
def is_self_crossing_simulation(distance):
    segs = segments_of(distance)
    for i in range(len(segs)):
        for j in range(i - 2):                    # j = 0 .. i-3
            if segments_intersect(segs[i], segs[j]):
                return True
    return False


def first_crossing(distance):
    # (i, [các j mà đoạn i chạm]) tại đoạn ĐẦU TIÊN gây cắt; None nếu không cắt.
    segs = segments_of(distance)
    for i in range(len(segs)):
        hits = [j for j in range(i - 2) if segments_intersect(segs[i], segs[j])]
        if hits:
            return i, hits
    return None


for d in ([2, 1, 1, 2], [1, 2, 3, 4], [1, 1, 1, 2, 1], [1, 1, 2, 2, 1, 1]):
    print(d, "->", is_self_crossing_simulation(d), "| first_crossing:", first_crossing(d))

[2, 1, 1, 2] -> True | first_crossing: (3, [0])
[1, 2, 3, 4] -> False | first_crossing: None
[1, 1, 1, 2, 1] -> True | first_crossing: (3, [0])
[1, 1, 2, 2, 1, 1] -> True | first_crossing: (5, [0])


### 4.6 Trace Cách 2 với `[1, 1, 2, 2, 1, 1]`

Chọn input này vì phải tới đoạn **cuối cùng** mới cắt, nên thấy được nhiều phép so sánh "không giao". Các điểm: `(0,0) → (0,1) → (-1,1) → (-1,-1) → (1,-1) → (1,0) → (0,0)`.

| `i` | đoạn `i` | so với `j` | đoạn `j` | chồng `x`? | chồng `y`? | Giao? |
|---:|---|---:|---|---|---|---|
| 3 | `(-1,-1)→(1,-1)` | 0 | `(0,0)→(0,1)` | có (`0` ∈ `[-1,1]`) | không (`-1` ∉ `[0,1]`) | không |
| 4 | `(1,-1)→(1,0)` | 0 | `(0,0)→(0,1)` | không (`1 ≠ 0`) | có | không |
| 4 | `(1,-1)→(1,0)` | 1 | `(0,1)→(-1,1)` | không | không | không |
| 5 | `(1,0)→(0,0)` | 0 | `(0,0)→(0,1)` | có (`[0,0]`) | có (`[0,0]`) | **có** → `True` |

Đoạn 5 đi sang trái trên `y = 0` và kết thúc đúng tại `(0, 0)` — chạm đầu dưới của đoạn 0. (Đoạn 0, 1, 2 không có gì để so vì `range(i - 2)` rỗng.)

In [14]:
def trace_simulation(distance):
    segs = segments_of(distance)
    for i in range(len(segs)):
        for j in range(i - 2):
            (a, b), (c, e) = segs[i], segs[j]
            ox = overlap(a[0], b[0], c[0], e[0])
            oy = overlap(a[1], b[1], c[1], e[1])
            print(f"i={i} {a}->{b} vs j={j} {c}->{e}: chồng x={ox}, chồng y={oy}")
            if ox and oy:
                print(f"=> đoạn {i} chạm đoạn {j}, trả về True")
                return True
    print("=> không có cặp nào giao, trả về False")
    return False


trace_simulation([1, 1, 2, 2, 1, 1])

i=3 (-1, -1)->(1, -1) vs j=0 (0, 0)->(0, 1): chồng x=True, chồng y=False
i=4 (1, -1)->(1, 0) vs j=0 (0, 0)->(0, 1): chồng x=False, chồng y=True
i=4 (1, -1)->(1, 0) vs j=1 (0, 1)->(-1, 1): chồng x=False, chồng y=False
i=5 (1, 0)->(0, 0) vs j=0 (0, 0)->(0, 1): chồng x=True, chồng y=True
=> đoạn 5 chạm đoạn 0, trả về True


True

### 4.7 Cách 2 vướng ở đâu?

- Thời gian: với mỗi `i`, ta so với khoảng `i - 2` đoạn cũ → tổng cộng khoảng `n²/2` phép so sánh → `O(n²)`. **Không phụ thuộc** độ dài các đoạn nữa — đã tiến bộ lớn so với Cách 1.
- Bộ nhớ: `O(n)` để lưu danh sách đoạn.

Nhưng với `n = 10^5`, `n²/2 = 5·10^9` phép so sánh — Python cần hàng giờ. Vẫn **TLE** (Time Limit Exceeded).

Câu hỏi dẫn tới lời giải chính:

> Đoạn mới có thật sự cần so với **mọi** đoạn cũ không? Một đoạn vẽ từ rất lâu (ví dụ đoạn 0 khi ta đang ở đoạn 1000) liệu có còn "với tới" được không?

Ta sẽ không đoán mò, mà dùng trọng tài để **làm thí nghiệm**.

## 5. Bước đột phá — chỉ cần nhìn 3 đoạn gần nhất

### 5.1 Thí nghiệm: tại lần cắt đầu tiên, đoạn bị đụng cách bao xa?

Với mỗi đường đi có cắt, gọi `first_crossing` để lấy đoạn đầu tiên gây cắt `i` và danh sách đoạn bị đụng `hits`. Gọi **khoảng cách** (gap) là `i - j`. Ta thống kê:

1. Gap **nhỏ nhất** trong `hits` thường là bao nhiêu?
2. Có trường hợp nào đoạn `i` **chỉ** đụng các đoạn rất xa (gap ≥ 6) mà **không** đụng đoạn nào có gap 3, 4 hay 5 không?
3. Tại lần cắt đầu tiên, `d[i-1] <= d[i-3]` có luôn đúng không? (Ý nghĩa của câu hỏi này sẽ rõ ở mục 5.6.)

Dữ liệu thí nghiệm: **vét cạn** mọi mảng độ dài 1–7 với giá trị 1–4 (21 844 mảng), cộng 3000 mảng ngẫu nhiên dài 6–16 (seed cố định). Bộ test `TESTS` này sẽ được dùng lại để đối chiếu chéo các cách giải.

In [15]:
import itertools
from collections import Counter


def build_tests():
    tests = []
    for n in range(1, 8):                                   # vét cạn: độ dài 1..7, giá trị 1..4
        for t in itertools.product(range(1, 5), repeat=n):
            tests.append(list(t))
    random.seed(335)
    for _ in range(3000):                                   # ngẫu nhiên: dài hơn, giá trị đa dạng
        n = random.randint(6, 16)
        hi = random.choice([3, 6, 10])
        tests.append([random.randint(1, hi) for _ in range(n)])
    return tests


TESTS = build_tests()
min_gap = Counter()
far_only = 0
start_outside = 0
crossing_count = 0
for d in TESTS:
    fc = first_crossing(d)
    if fc is None:
        continue
    crossing_count += 1
    i, hits = fc
    gaps = [i - j for j in hits]
    min_gap[min(gaps)] += 1
    if not any(g in (3, 4, 5) for g in gaps):
        far_only += 1
    if d[i - 1] > d[i - 3]:
        start_outside += 1

print("số test:", len(TESTS), "| số test có cắt:", crossing_count)
print("gap nhỏ nhất tại lần cắt đầu tiên:", sorted(min_gap.items()))
print("số lần chỉ đụng đoạn xa (gap >= 6) mà không đụng gap 3/4/5:", far_only)
print("số lần d[i-1] > d[i-3] tại lần cắt đầu tiên:", start_outside)

số test: 24844 | số test có cắt: 23603
gap nhỏ nhất tại lần cắt đầu tiên: [(3, 21549), (4, 1356), (5, 698)]
số lần chỉ đụng đoạn xa (gap >= 6) mà không đụng gap 3/4/5: 0
số lần d[i-1] > d[i-3] tại lần cắt đầu tiên: 0


### 5.2 Kết quả thí nghiệm nói gì?

- Gap nhỏ nhất **chỉ** có thể là `3`, `4` hoặc `5`. Không bao giờ là `6`, `7`, …
- **Không có** lần nào đoạn `i` chỉ đụng đoạn xa mà bỏ qua các đoạn gần. (Đoạn `i` *có thể* đụng thêm `i-6`, `i-7`… nhưng khi đó nó **luôn** đụng kèm một trong `i-3`, `i-4`, `i-5`.)
- Tại lần cắt đầu tiên, luôn có `d[i-1] <= d[i-3]`.

Như vậy, để biết đường đi có tự cắt hay không, khi vẽ đoạn `i` ta chỉ cần hỏi: **đoạn `i` có đụng `i-3`, `i-4` hay `i-5` không?** Mỗi câu hỏi chỉ là vài phép so sánh độ dài — `O(1)` — nên cả thuật toán là `O(n)`.

Nhớ lại quy tắc chẵn/lẻ ở mục 2.1:

| Đoạn bị đụng | Gap | Quan hệ với đoạn `i` | Kiểu đụng |
|---|---:|---|---|
| `i-3` | 3 (lẻ) | vuông góc | đoạn `i` **cắt ngang** (hoặc chạm) `i-3` — **Trường hợp 1** |
| `i-4` | 4 (chẵn) | song song **cùng chiều** | đoạn `i` đi **trên cùng đường thẳng** và chạm đầu `i-4` — **Trường hợp 2** |
| `i-5` | 5 (lẻ) | vuông góc | đoạn `i` **cắt ngang** (hoặc chạm) `i-5` — **Trường hợp 3** |

Ba mục tiếp theo phân tích từng trường hợp: vẽ hình, đặt tọa độ, rồi rút ra điều kiện bằng **độ dài**. Với mỗi trường hợp, ta xoay hình sao cho khớp với ví dụ nhỏ nhất (đoạn sớm nhất đi lên Bắc), để hình trong markdown giống hệt hình `draw_path` vẽ.

### 5.3 Trường hợp 1 — đoạn `i` cắt đoạn `i-3`

Bốn đoạn liên tiếp `i-3` (lên), `i-2` (sang trái), `i-1` (xuống), `i` (sang phải):

```text
               i-2
        +<-------------+
        |              ^
   i-1  |              |
        v              |  i-3
        +--------------X------>   i
                       |
                       o   (điểm đầu của i-3)
```

Đoạn `i` đi sang phải và **đâm vào** bức tường `i-3` khi thỏa đồng thời:

1. **`d[i] >= d[i-2]`** — đi đủ xa sang phải: đoạn `i-2` đã đi sang trái `d[i-2]` mét tính từ đường thẳng của `i-3`, nên đoạn `i` phải đi ít nhất từng ấy mét mới quay lại được đường thẳng đó.
2. **`d[i-1] <= d[i-3]`** — đoạn `i` nằm **không thấp hơn** điểm đầu `o` của `i-3`: đoạn `i-1` đi xuống `d[i-1]` mét từ đỉnh, mà `i-3` chỉ cao `d[i-3]` mét. Nếu `d[i-1] > d[i-3]` thì đoạn `i` chạy **bên dưới** `o`, lướt qua mà không chạm.

Dấu `=` ở cả hai chỗ là các trường hợp **chạm**: `d[i] == d[i-2]` → đầu mút của `i` vừa chạm vào `i-3`; `d[i-1] == d[i-3]` → đoạn `i` đi qua đúng điểm `o`.

Ví dụ số, Example 1 `[2, 1, 1, 2]`, `i = 3`: `d[3] = 2 >= d[1] = 1` ✓ và `d[2] = 1 <= d[0] = 2` ✓ → cắt.

In [16]:
def yn(b):
    return "✓" if b else "✗"


def case1(d, i):
    return d[i] >= d[i - 2] and d[i - 1] <= d[i - 3]


for d, i in (([2, 1, 1, 2], 3), ([1, 1, 1, 1], 3), ([2, 3, 2, 4], 3), ([1, 2, 3, 4], 3)):
    print(f"distance = {d}, i = {i}: d[i]={d[i]} >= d[i-2]={d[i-2]} {yn(d[i] >= d[i-2])}, "
          f"d[i-1]={d[i-1]} <= d[i-3]={d[i-3]} {yn(d[i-1] <= d[i-3])} -> TH1 = {case1(d, i)}")
    draw_path(d)
    print()

distance = [2, 1, 1, 2], i = 3: d[i]=2 >= d[i-2]=1 ✓, d[i-1]=1 <= d[i-3]=2 ✓ -> TH1 = True
+11+
2  0
+33X33*
   0
   S

distance = [1, 1, 1, 1], i = 3: d[i]=1 >= d[i-2]=1 ✓, d[i-1]=1 <= d[i-3]=1 ✓ -> TH1 = True
+11+
2  0
+33X

distance = [2, 3, 2, 4], i = 3: d[i]=4 >= d[i-2]=3 ✓, d[i-1]=2 <= d[i-3]=2 ✓ -> TH1 = True
+11111111+
2        0
2        0
2        0
+33333333X33*

distance = [1, 2, 3, 4], i = 3: d[i]=4 >= d[i-2]=2 ✓, d[i-1]=3 <= d[i-3]=1 ✗ -> TH1 = False
+11111+
2     0
2     S
2
2
2
+33333333333*



Bốn hình trên cho thấy:

- `[2, 1, 1, 2]`: cắt chữ thập điển hình.
- `[1, 1, 1, 1]`: cả hai điều kiện đều xảy ra dấu `=` (`1 >= 1`, `1 <= 1`) → đoạn 3 kết thúc **đúng tại** `o = (0, 0)`. Chỉ là chạm, nhưng vẫn là cắt.
- `[2, 3, 2, 4]`: `d[i-1] = 2 == d[i-3] = 2` → đoạn 3 đi qua đúng điểm `o` rồi đi tiếp. Vẫn cắt.
- `[1, 2, 3, 4]` (Example 2): điều kiện 1 đúng (`4 >= 2`) nhưng điều kiện 2 sai (`3 <= 1` sai) → đoạn 3 chạy bên dưới `o`, không chạm. Đây chính là đường **xoắn ra**.

### 5.4 Trường hợp 2 — đoạn `i` chạm đoạn `i-4` (cùng một đường thẳng)

Năm đoạn `i-4` (lên), `i-3` (trái), `i-2` (xuống), `i-1` (phải), `i` (lên):

```text
            i-3
       +<----------+
       |           ^
       |           |  i-4
  i-2  |           o   (điểm đầu của i-4)
       |           ^
       |           |  i     (cùng đường thẳng với i-4, đi lên chạm o)
       v           |
       +---------->+
            i-1
```

Đoạn `i` song song **cùng chiều** với `i-4`. Muốn chạm được nhau, hai đoạn phải nằm **trên cùng một đường thẳng**, rồi đoạn `i` phải đi lên đủ cao để chạm điểm đầu `o` của `i-4`:

1. **`d[i-1] == d[i-3]`** — đoạn `i-3` đi sang trái `d[i-3]` mét, đoạn `i-1` đi sang phải `d[i-1]` mét; bằng nhau thì đoạn `i` quay về **đúng** đường thẳng của `i-4`. (Nếu `d[i-1] < d[i-3]`, đoạn `i` nằm bên trái `i-4`, không thể chạm `i-4`; nếu `d[i-1] > d[i-3]` thì nằm bên phải.)
2. **`d[i] + d[i-4] >= d[i-2]`** — đoạn `i-2` đi xuống `d[i-2]` mét tính từ đỉnh của `i-4`; điểm `o` thấp hơn đỉnh `d[i-4]` mét. Vậy khoảng cách từ đáy (điểm đầu của `i`) lên tới `o` là `d[i-2] - d[i-4]`, và đoạn `i` chạm `o` khi `d[i] >= d[i-2] - d[i-4]`, tức `d[i] + d[i-4] >= d[i-2]`.

Ví dụ số `[1, 1, 2, 1, 1]`, `i = 4`: `d[3] = 1 == d[1] = 1` ✓ và `d[4] + d[0] = 2 >= d[2] = 2` ✓ → chạm tại `(0, 0)`.

Để ý: Trường hợp 1 **không** bắt được ca này. Ở `[1, 1, 2, 1, 1]`, đoạn `i` đi lên về phía cái "trần" `i-3`, nhưng `d[4] = 1 >= d[2] = 2` sai — nó **chưa lên tới trần** thì đã đụng đầu dưới của `i-4` rồi. Vì vậy cần một điều kiện riêng cho kiểu đụng "đối đầu trên cùng đường thẳng" này.

In [17]:
def case2(d, i):
    return i >= 4 and d[i - 1] == d[i - 3] and d[i] + d[i - 4] >= d[i - 2]


for d, i in (([1, 1, 2, 1, 1], 4), ([2, 3, 4, 3, 3, 1], 4), ([1, 1, 3, 1, 1], 4)):
    print(f"distance = {d}, i = {i}: d[i-1]={d[i-1]} == d[i-3]={d[i-3]} {yn(d[i-1] == d[i-3])}, "
          f"d[i]+d[i-4]={d[i] + d[i-4]} >= d[i-2]={d[i-2]} {yn(d[i] + d[i-4] >= d[i-2])}"
          f" -> TH1 = {case1(d, i)}, TH2 = {case2(d, i)}")
    draw_path(d)
    print()

distance = [1, 1, 2, 1, 1], i = 4: d[i-1]=1 == d[i-3]=1 ✓, d[i]+d[i-4]=2 >= d[i-2]=2 ✓ -> TH1 = False, TH2 = True
+11+
2  0
2  X
2  4
+33+

distance = [2, 3, 4, 3, 3, 1], i = 4: d[i-1]=3 == d[i-3]=3 ✓, d[i]+d[i-4]=5 >= d[i-2]=4 ✓ -> TH1 = False, TH2 = True
+11111111+
2        0
2     *55X
2        X
2        X
2        4
2        4
2        4
+33333333+

distance = [1, 1, 3, 1, 1], i = 4: d[i-1]=1 == d[i-3]=1 ✓, d[i]+d[i-4]=2 >= d[i-2]=3 ✗ -> TH1 = False, TH2 = False
+11+
2  0
2  S
2
2  *
2  4
+33+



- `[1, 1, 2, 1, 1]`: đoạn 4 đi lên và **chạm đúng** điểm xuất phát.
- `[2, 3, 4, 3, 3, 1]`: đoạn 4 đi lên `3` mét, vượt qua `o` (cần `4 - 2 = 2` mét) và **đè lên** một phần của đoạn 0 (cột `X`). Đoạn 5 phía sau không còn quan trọng — đã cắt ở đoạn 4.
- `[1, 1, 3, 1, 1]`: cùng đường thẳng (`1 == 1`) nhưng đoạn 4 chỉ đi `1` mét, còn cần `3 - 1 = 2` mét mới tới `o` → **không** chạm.

### 5.5 Trường hợp 3 — đoạn `i` cắt đoạn `i-5`

Sáu đoạn `i-5` (lên), `i-4` (trái), `i-3` (xuống), `i-2` (phải), `i-1` (lên), `i` (trái):

```text
               i-4
        +<-----------+
        |            ^
        |            |  i-5
   i-3  |     <------X--------+    <- đoạn i đi sang trái, cắt i-5
        |      i     |        ^
        |            o        |  i-1
        |   (điểm đầu i-5)    |
        v                     |
        +-------------------->+
                 i-2
```

Đây là trường hợp "lọt vào khe": đoạn `i-5` thò xuống bên trong cái hộp tạo bởi `i-4`, `i-3`, `i-2`, `i-1`, và đoạn `i` đâm ngang vào nó. Cần **cả bốn** điều kiện:

1. **`d[i-2] >= d[i-4]`** — đáy `i-2` đi sang phải đủ xa để vượt qua (hoặc tới) đường thẳng của `i-5`. Nhờ vậy đoạn `i-1` nằm **bên phải** `i-5`, và đoạn `i` đi sang trái mới có cơ hội gặp `i-5`.
2. **`d[i] + d[i-4] >= d[i-2]`** — đoạn `i` đi sang trái đủ xa: nó xuất phát cách đường thẳng của `i-5` đúng `d[i-2] - d[i-4]` mét, nên cần `d[i] >= d[i-2] - d[i-4]`.
3. **`d[i-1] + d[i-5] >= d[i-3]`** — đoạn `i` nằm **không thấp hơn** điểm đầu `o` của `i-5`. Đáy cách trần `d[i-3]` mét; đoạn `i` cao hơn đáy `d[i-1]` mét; điểm `o` thấp hơn trần `d[i-5]` mét, tức cao hơn đáy `d[i-3] - d[i-5]` mét. Cần `d[i-1] >= d[i-3] - d[i-5]`.
4. **`d[i-1] <= d[i-3]`** — đoạn `i` nằm **không cao hơn** trần `i-4`. Nếu `i-1` vọt lên cao hơn trần, đoạn `i` chạy ra **ngoài hộp**, bên trên mọi thứ, và chẳng cắt gì cả.

Ví dụ số `[1, 1, 2, 2, 1, 1]`, `i = 5`: (1) `d[3] = 2 >= d[1] = 1` ✓, (2) `d[5] + d[1] = 2 >= d[3] = 2` ✓, (3) `d[4] + d[0] = 2 >= d[2] = 2` ✓, (4) `d[4] = 1 <= d[2] = 2` ✓ → cắt (thực ra là chạm đúng điểm `o = (0, 0)`).

Điều kiện (4) rất hay bị quên vì "trông có vẻ thừa". Hình thứ ba bên dưới cho thấy nó **không** thừa.

In [18]:
def case3(d, i):
    return (i >= 5 and d[i - 2] >= d[i - 4] and d[i] + d[i - 4] >= d[i - 2]
            and d[i - 1] + d[i - 5] >= d[i - 3] and d[i - 1] <= d[i - 3])


for d in ([1, 1, 2, 2, 1, 1], [3, 4, 5, 6, 3, 3], [1, 1, 2, 2, 3, 1]):
    i = 5
    c = [d[i-2] >= d[i-4], d[i] + d[i-4] >= d[i-2], d[i-1] + d[i-5] >= d[i-3], d[i-1] <= d[i-3]]
    print(f"distance = {d}, i = 5: điều kiện (1)..(4) = {' '.join(yn(b) for b in c)}"
          f" -> TH3 = {case3(d, i)}, trọng tài: {is_self_crossing_simulation(d)}")
    draw_path(d)
    print()

distance = [1, 1, 2, 2, 1, 1], i = 5: điều kiện (1)..(4) = ✓ ✓ ✓ ✓ -> TH3 = True, trọng tài: True
+11+
2  0
2  X55+
2     4
+33333+

distance = [3, 4, 5, 6, 3, 3], i = 5: điều kiện (1)..(4) = ✓ ✓ ✓ ✓ -> TH3 = True, trọng tài: True
+11111111111+
2           0
2           0
2           0
2        *55X55555+
2           0     4
2           S     4
2                 4
2                 4
2                 4
+33333333333333333+

distance = [1, 1, 2, 2, 3, 1], i = 5: điều kiện (1)..(4) = ✓ ✓ ✓ ✗ -> TH3 = False, trọng tài: False
   *55+
      4
+11+  4
2  0  4
2  S  4
2     4
+33333+



- `[1, 1, 2, 2, 1, 1]`: đoạn 5 chạm đúng điểm xuất phát (chữ `X` ở đầu dưới đoạn 0).
- `[3, 4, 5, 6, 3, 3]`: đoạn 5 cắt **xuyên** đoạn 0 tại `(0, 1)`, ở giữa thân đoạn 0.
- `[1, 1, 2, 2, 3, 1]`: ba điều kiện đầu đều đúng, chỉ điều kiện (4) sai (`3 <= 2` sai). Đoạn 4 vọt lên cao hơn trần, đoạn 5 chạy **bên trên** cả hình → không cắt. Nếu quên điều kiện (4), ta sẽ trả `True` sai.

### 5.6 Vì sao không bao giờ cần xét `i-6`, `i-7`, …?

Thí nghiệm đã cho thấy điều đó; giờ ta hiểu **tại sao** bằng một hình. Giả sử đường đi **chưa** cắt trước đoạn `i`. Xoay hình như Trường hợp 1: `i-3` đi lên, `i-2` sang trái, `i-1` xuống, đoạn `i` đi sang phải từ điểm `D`.

```text
                    i-2
       C +<----------------------+ B
         |                       ^
         |                       |
    i-1  |                       |  i-3
         |      i-5              |
         v       |               |
       D +---->  |               |        <- đoạn i (đi sang phải)
                 |               |
                 v               |
   - - - - - - - +-------------->+ A - - - -    (đường thẳng chứa i-4)
                       i-4
```

Ba đoạn `i-1`, `i-2`, `i-3` tạo thành một **chữ U** úp ngược, miệng hở ở phía dưới, và đoạn `i-4` nằm trên đường thẳng của miệng (nó đi sang phải và kết thúc ở `A`, chân của `i-3`). Có hai khả năng:

**(a) `d[i-1] > d[i-3]`: điểm `D` nằm thấp hơn `A`, tức là BÊN NGOÀI chữ U.** Lúc này đường đi đang **xoắn ra**: đoạn `i` chạy bên dưới toàn bộ phần đã vẽ, vào vùng hoàn toàn trống → không thể cắt gì. (Đây là kết quả thứ 3 của thí nghiệm: tại lần cắt đầu tiên **luôn** có `d[i-1] <= d[i-3]`. Cả ba trường hợp cắt đều chứa điều kiện này.)

**(b) `d[i-1] <= d[i-3]`: điểm `D` nằm BÊN TRONG chữ U (hoặc đúng trên miệng).** Đoạn `i` bị nhốt trong chữ U. Nó có thể đụng:

- bức tường phải `i-3` → **Trường hợp 1**;
- đoạn `i-4` nằm trên miệng — chỉ khi `D` nằm **đúng** trên miệng (`d[i-1] == d[i-3]`) → **Trường hợp 2**;
- đoạn `i-5` — nó nối vào đầu trái của `i-4` và có thể **thò vào** trong chữ U → **Trường hợp 3**.

Còn các đoạn cũ hơn (`i-6`, `i-7`, …)? Chúng được vẽ **trước** chữ U. Muốn nằm trong vùng đoạn `i` đi qua, chúng phải hoặc **xuyên qua** tường `i-1`, `i-2`, `i-3` (nhưng khi vẽ các tường đó ta đã kiểm tra và phát hiện cắt rồi — trái với giả sử "chưa cắt"), hoặc chui vào qua chỗ `i-5` thò vào. Trường hợp sau, chúng nối tiếp với đỉnh của `i-5` nên nằm **sau lưng** `i-5` so với hướng đi của đoạn `i` (đoạn `i` đến từ bên trái): muốn chạm chúng, đoạn `i` phải đi qua đường thẳng của `i-5` ở đúng khúc `i-5` đang đứng, tức là **đã chạm `i-5` trước** — và Trường hợp 3 đã bắt được.

> Lập luận trên là trực giác hình học (một chứng minh chặt chẽ phải xét thêm vài cấu hình con). Bù lại, ta đã kiểm chứng bằng **vét cạn** mọi mảng nhỏ và hàng nghìn mảng ngẫu nhiên ở mục 5.1, và sẽ còn đối chiếu chéo lần nữa ở mục 7.

## 6. Cách 3 — Lời giải chính: kiểm tra 3 trường hợp, `O(n)` / `O(1)`

### 6.1 Ghép lại

```text
với mỗi i = 3, 4, …, n-1:
    TH1 (i cắt i-3):              d[i] >= d[i-2]  và  d[i-1] <= d[i-3]
    TH2 (i chạm i-4), cần i >= 4: d[i-1] == d[i-3]  và  d[i] + d[i-4] >= d[i-2]
    TH3 (i cắt i-5), cần i >= 5:  d[i-2] >= d[i-4]  và  d[i] + d[i-4] >= d[i-2]
                                  và  d[i-1] + d[i-5] >= d[i-3]  và  d[i-1] <= d[i-3]
    nếu một trong ba đúng: trả True
trả False
```

Vì sao bắt đầu từ `i = 3`? Ba đoạn đầu (`0, 1, 2`) không thể cắt nhau (đoạn 2 chỉ có thể so với đoạn ≤ `-1`). Đoạn 3 là đoạn đầu tiên có "đối thủ" `i-3 = 0`.

**Cảnh báo về chỉ số âm.** Trong Python, `d[-1]` **không** báo lỗi mà trả về **phần tử cuối** của list! Nếu viết TH2 mà quên `i >= 4`, thì ở `i = 3` biểu thức `d[i-4]` thành `d[-1]` — một giá trị chẳng liên quan — và có thể báo cắt sai. Vì thế điều kiện `i >= 4` / `i >= 5` phải đứng **đầu tiên** trong phép `and` (Python dừng đánh giá `and` ngay khi gặp vế sai — gọi là *short-circuit*). Mục 10 sẽ cho một phản ví dụ cụ thể.

In [19]:
def is_self_crossing_cases(distance):
    d = distance
    for i in range(3, len(d)):
        # TH1: đoạn i cắt / chạm đoạn i-3
        if d[i] >= d[i - 2] and d[i - 1] <= d[i - 3]:
            return True
        # TH2: đoạn i chạm đoạn i-4 trên cùng một đường thẳng
        if i >= 4 and d[i - 1] == d[i - 3] and d[i] + d[i - 4] >= d[i - 2]:
            return True
        # TH3: đoạn i cắt / chạm đoạn i-5
        if (i >= 5 and d[i - 2] >= d[i - 4] and d[i] + d[i - 4] >= d[i - 2]
                and d[i - 1] + d[i - 5] >= d[i - 3] and d[i - 1] <= d[i - 3]):
            return True
    return False


for d in ([2, 1, 1, 2], [1, 2, 3, 4], [1, 1, 1, 2, 1], [1, 1, 2, 1, 1], [1, 1, 2, 2, 1, 1], [1, 1, 2, 2, 3, 1]):
    print(d, "->", is_self_crossing_cases(d))

[2, 1, 1, 2] -> True
[1, 2, 3, 4] -> False
[1, 1, 1, 2, 1] -> True
[1, 1, 2, 1, 1] -> True
[1, 1, 2, 2, 1, 1] -> True
[1, 1, 2, 2, 3, 1] -> False


### 6.2 Trace Cách 3

Input đầu tiên `[3, 4, 5, 6, 3, 3]` đi qua đủ ba trường hợp trước khi bắt được TH3 ở `i = 5`; input thứ hai `[1, 1, 2, 2, 3, 1]` là ca "bẫy" của điều kiện (4).

**`d = [3, 4, 5, 6, 3, 3]`**

| `i` | TH1 | TH2 | TH3 | Kết luận |
|---:|---|---|---|---|
| 3 | `6 >= 4` ✓, `5 <= 3` ✗ | (cần `i >= 4`) | (cần `i >= 5`) | đi tiếp |
| 4 | `3 >= 5` ✗, `6 <= 4` ✗ | `6 == 4` ✗, `3+3 >= 5` ✓ | (cần `i >= 5`) | đi tiếp |
| 5 | `3 >= 6` ✗, `3 <= 5` ✓ | `3 == 5` ✗, `3+4 >= 6` ✓ | `6 >= 4` ✓, `3+4 >= 6` ✓, `3+3 >= 5` ✓, `3 <= 5` ✓ | **`True`** |

**`d = [1, 1, 2, 2, 3, 1]`**

| `i` | TH1 | TH2 | TH3 | Kết luận |
|---:|---|---|---|---|
| 3 | `2 >= 1` ✓, `2 <= 1` ✗ | — | — | đi tiếp |
| 4 | `3 >= 2` ✓, `2 <= 1` ✗ | `2 == 1` ✗, `3+1 >= 2` ✓ | — | đi tiếp |
| 5 | `1 >= 2` ✗, `3 <= 2` ✗ | `3 == 2` ✗, `1+1 >= 2` ✓ | `2 >= 1` ✓, `1+1 >= 2` ✓, `3+1 >= 2` ✓, `3 <= 2` ✗ | hết mảng → **`False`** |

Mỗi trường hợp chỉ đúng khi **mọi** vế của nó đúng; một vế ✗ là đủ loại trường hợp đó. Thứ tự bốn vế của TH3 trong bảng là (1) `d[i-2] >= d[i-4]`, (2) `d[i] + d[i-4] >= d[i-2]`, (3) `d[i-1] + d[i-5] >= d[i-3]`, (4) `d[i-1] <= d[i-3]`, giống mục 5.5. (Khi chạy thật, Python dừng phép `and` ngay ở vế sai đầu tiên; hàm trace in đủ các vế chỉ để ta quan sát.)

In [20]:
def trace_cases(distance):
    d = distance
    print("d =", d)
    for i in range(3, len(d)):
        th1 = d[i] >= d[i - 2] and d[i - 1] <= d[i - 3]
        print(f"  i={i}: TH1 {d[i]}>={d[i-2]} {yn(d[i] >= d[i-2])}, {d[i-1]}<={d[i-3]} {yn(d[i-1] <= d[i-3])}"
              f" -> {th1}")
        th2 = False
        if i >= 4:
            th2 = d[i - 1] == d[i - 3] and d[i] + d[i - 4] >= d[i - 2]
            print(f"        TH2 {d[i-1]}=={d[i-3]} {yn(d[i-1] == d[i-3])}, "
                  f"{d[i]}+{d[i-4]}>={d[i-2]} {yn(d[i] + d[i-4] >= d[i-2])} -> {th2}")
        th3 = False
        if i >= 5:
            c = [d[i-2] >= d[i-4], d[i] + d[i-4] >= d[i-2], d[i-1] + d[i-5] >= d[i-3], d[i-1] <= d[i-3]]
            th3 = all(c)
            print(f"        TH3 {d[i-2]}>={d[i-4]} {yn(c[0])}, {d[i]}+{d[i-4]}>={d[i-2]} {yn(c[1])}, "
                  f"{d[i-1]}+{d[i-5]}>={d[i-3]} {yn(c[2])}, {d[i-1]}<={d[i-3]} {yn(c[3])} -> {th3}")
        if th1 or th2 or th3:
            print(f"  => cắt tại đoạn {i}, trả về True")
            return True
    print("  => duyệt hết, trả về False")
    return False


trace_cases([3, 4, 5, 6, 3, 3])
print()
trace_cases([1, 1, 2, 2, 3, 1])

d = [3, 4, 5, 6, 3, 3]
  i=3: TH1 6>=4 ✓, 5<=3 ✗ -> False
  i=4: TH1 3>=5 ✗, 6<=4 ✗ -> False
        TH2 6==4 ✗, 3+3>=5 ✓ -> False
  i=5: TH1 3>=6 ✗, 3<=5 ✓ -> False
        TH2 3==5 ✗, 3+4>=6 ✓ -> False
        TH3 6>=4 ✓, 3+4>=6 ✓, 3+3>=5 ✓, 3<=5 ✓ -> True
  => cắt tại đoạn 5, trả về True

d = [1, 1, 2, 2, 3, 1]
  i=3: TH1 2>=1 ✓, 2<=1 ✗ -> False
  i=4: TH1 3>=2 ✓, 2<=1 ✗ -> False
        TH2 2==1 ✗, 3+1>=2 ✓ -> False
  i=5: TH1 1>=2 ✗, 3<=2 ✗ -> False
        TH2 3==2 ✗, 1+1>=2 ✓ -> False
        TH3 2>=1 ✓, 1+1>=2 ✓, 3+1>=2 ✓, 3<=2 ✗ -> False
  => duyệt hết, trả về False


False

## 7. Code nộp LeetCode

Chuyển `is_self_crossing_cases` vào `class Solution` với đúng signature của đề. Trên LeetCode, `List` đã được import sẵn; khi chạy ngoài LeetCode ta cần `from typing import List`.

Bộ test gồm: 3 ví dụ của đề, các ca biên (1, 2, 3 đoạn), các ca **chạm** (`[1,1,1,1]`), mỗi trường hợp TH1/TH2/TH3, ca bẫy điều kiện (4), ca bẫy **chỉ số âm** `[1, 2, 1, 1]`, đường xoắn ra/xoắn vào, và hai input cỡ lớn `n = 10^5`.

In [21]:
from typing import List


class Solution:
    def isSelfCrossing(self, distance: List[int]) -> bool:
        d = distance
        for i in range(3, len(d)):
            # TH1: đoạn i cắt / chạm đoạn i-3
            if d[i] >= d[i - 2] and d[i - 1] <= d[i - 3]:
                return True
            # TH2: đoạn i chạm đoạn i-4 (cùng một đường thẳng)
            if i >= 4 and d[i - 1] == d[i - 3] and d[i] + d[i - 4] >= d[i - 2]:
                return True
            # TH3: đoạn i cắt / chạm đoạn i-5
            if (i >= 5 and d[i - 2] >= d[i - 4] and d[i] + d[i - 4] >= d[i - 2]
                    and d[i - 1] + d[i - 5] >= d[i - 3] and d[i - 1] <= d[i - 3]):
                return True
        return False


sol = Solution()
assert sol.isSelfCrossing([2, 1, 1, 2]) is True          # Example 1
assert sol.isSelfCrossing([1, 2, 3, 4]) is False         # Example 2
assert sol.isSelfCrossing([1, 1, 1, 2, 1]) is True       # Example 3
assert sol.isSelfCrossing([1]) is False
assert sol.isSelfCrossing([1, 1]) is False
assert sol.isSelfCrossing([1, 1, 1]) is False
assert sol.isSelfCrossing([1, 1, 1, 1]) is True          # quay về đúng gốc: chạm
assert sol.isSelfCrossing([1, 1, 2, 1, 1]) is True       # TH2
assert sol.isSelfCrossing([1, 1, 2, 2, 1, 1]) is True    # TH3 (chạm)
assert sol.isSelfCrossing([3, 4, 5, 6, 3, 3]) is True    # TH3 (cắt xuyên)
assert sol.isSelfCrossing([1, 1, 2, 2, 3, 1]) is False   # bẫy điều kiện (4) của TH3
assert sol.isSelfCrossing([1, 2, 1, 1]) is False         # bẫy chỉ số âm
assert sol.isSelfCrossing([1, 2, 3, 4, 5, 6]) is False   # xoắn ra
assert sol.isSelfCrossing([6, 5, 4, 3, 2, 1]) is False   # xoắn vào
assert sol.isSelfCrossing(list(range(1, 10**5 + 1))) is False   # n = 10^5, xoắn ra mãi
assert sol.isSelfCrossing([10**5] * 10**5) is True              # cắt ngay ở đoạn 3
print("Tất cả test của Solution đều PASS")

Tất cả test của Solution đều PASS


### 7.1 Đối chiếu chéo với trọng tài

Một thuật toán toàn phép so sánh như thế này rất dễ sai một dấu. Cách đáng tin nhất là so nó với **hai** lời giải chậm nhưng chắc chắn đúng (Cách 1 đi từng mét, Cách 2 mô phỏng đoạn thẳng) trên toàn bộ `TESTS` (vét cạn + ngẫu nhiên, seed cố định), cộng thêm 300 test ngẫu nhiên dài tới 60 đoạn.

In [22]:
random.seed(42)
extra = [[random.randint(1, 8) for _ in range(random.randint(1, 60))] for _ in range(300)]

mismatch = 0
for d in TESTS + extra:
    ref = is_self_crossing_simulation(d)
    if not (is_self_crossing_grid(d) == ref == is_self_crossing_cases(d) == sol.isSelfCrossing(d)):
        mismatch += 1
        print("LỆCH:", d)
print(f"Đã so {len(TESTS) + len(extra)} test, số lần lệch: {mismatch}")
assert mismatch == 0

Đã so 25144 test, số lần lệch: 0


## 8. Cách 4 — Góc nhìn "xoắn ra / xoắn vào" (spiral out / spiral in)

Cách 3 đúng, nhưng ba trường hợp trông hơi "từ trên trời rơi xuống". Cách 4 kể lại câu chuyện bằng **hình dạng tổng thể** của đường đi, và cho một thuật toán khác cũng chạy `O(n)` thời gian.

### 8.1 Ba kiểu hình dạng

So sánh mỗi đoạn với đoạn **song song trước nó** (`d[i]` với `d[i-2]`):

- **Xoắn ra** (spiral out): `d[i] > d[i-2]` với mọi `i`. Mỗi đoạn dài hơn đoạn song song trước đó, nên nó luôn nằm trên một đường thẳng **mới, ở ngoài** toàn bộ phần đã vẽ. Ví dụ `[1, 2, 3, 4, 5, 6]`: các điểm `(0,0), (0,1), (-2,1), (-2,-2), (2,-2), (2,3), (-4,3)` — mỗi đường thẳng mới (`x = 2`, `y = 3`, `x = -4`…) đều nằm ngoài hình chữ nhật bao quanh các điểm cũ. Đoạn ở ngoài thì không thể đụng đoạn ở trong → **không bao giờ cắt**.
- **Xoắn vào** (spiral in): `d[i] < d[i-2]`. Mỗi đoạn ngắn hơn đoạn song song trước đó, đường đi cuộn dần vào trong. Một khi đã xoắn vào, mỗi đoạn mới **bắt buộc** phải ngắn hơn đoạn song song trước nó: nếu `d[i] >= d[i-2]` thì đoạn `i` đâm vào đoạn `i-3` (đó chính là TH1, vì khi đang xoắn vào thì `d[i-1] < d[i-3]` tự động đúng).
- **Xoắn ra rồi xoắn vào**: đi ra một lúc, rồi tại một đoạn nào đó lần đầu `d[i] <= d[i-2]` — gọi là **bước chuyển tiếp** — sau đó xoắn vào. Còn xoắn vào rồi lại xoắn ra thì **không thể** nếu không cắt (đoạn muốn ra ngoài phải đâm qua tường).

In [23]:
for d, label in (([1, 2, 3, 4, 5, 6], "xoắn ra"), ([6, 5, 4, 3, 2, 1], "xoắn vào"),
                 ([1, 1, 2, 2, 4, 1, 1], "xoắn ra rồi xoắn vào")):
    print(f"{d}: {label}, cắt? {is_self_crossing_simulation(d)}")
    draw_path(d)
    print()

[1, 2, 3, 4, 5, 6]: xoắn ra, cắt? False
*55555555555555555+
                  4
                  4
                  4
      +11111+     4
      2     0     4
      2     S     4
      2           4
      2           4
      2           4
      +33333333333+

[6, 5, 4, 3, 2, 1]: xoắn vào, cắt? False
+11111111111111+
2              0
2              0
2              0
2     *55+     0
2        4     0
2        4     0
2        4     0
+33333333+     0
               0
               0
               0
               S

[1, 1, 2, 2, 4, 1, 1]: xoắn ra rồi xoắn vào, cắt? False
   +55+
   6  4
   *  4
      4
+11+  4
2  0  4
2  S  4
2     4
+33333+



### 8.2 Bước chuyển tiếp: vì sao phải "thu ngắn" `d[i-1]`?

Thuật toán thô sơ:

```text
1. Xoắn ra: tăng i chừng nào d[i] > d[i-2].
2. Xoắn vào: từ i+1 trở đi, nếu d[k] >= d[k-2] thì trả True.
```

Nó **sai** với `[1, 1, 2, 2, 1, 1]` (đáp án đúng là `True`, Trường hợp 3):

```text
+11+
2  0
2  X55+         đoạn 4 (đi lên) là bước chuyển tiếp: d[4] = 1 <= d[2] = 2
2     4         đoạn 5 (sang trái) chỉ dài 1 < d[3] = 2, thuật toán thô nghĩ "an toàn"
+33333+         nhưng nó đâm vào đoạn 0 trước khi tới được tường đoạn 2!
```

Vấn đề: ở bước chuyển tiếp `i`, nếu đoạn `i` **vươn lên đủ cao** — tới ngang tầm đoạn `i-4` — thì "bức tường" mà đoạn `i+1` phải tránh **không còn** là đoạn `i-2` ở xa (cách `d[i-1]` mét), mà là đoạn `i-4` / `i-5` ở gần hơn. Khoảng cách từ đường thẳng của đoạn `i` tới đường thẳng của `i-4` là `d[i-1] - d[i-3]`.

- "Vươn tới ngang tầm `i-4`" ⇔ `d[i] >= d[i-2] - d[i-4]` (cùng phép tính như điều kiện (2) của TH2/TH3).
- Khi đó ta **thu ngắn** chiều dài cho phép: `d[i-1] = d[i-1] - d[i-3]`. Từ bước sau trở đi, chỉ cần áp quy tắc xoắn vào bình thường.

Với `[1, 1, 2, 2, 1, 1]`: chuyển tiếp ở `i = 4`, `d[4] = 1 >= d[2] - d[0] = 1` ✓ → `d[3] = 2 - 1 = 1`. Sang `i = 5`: `d[5] = 1 >= d[3] = 1` → `True`. Đúng!

Chi tiết cài đặt:

- Khi `i` còn nhỏ (`i = 2` hoặc `i = 3`), đoạn `i-4` (và với `i = 2` cả đoạn `i-3`) chưa tồn tại → coi độ dài bằng `0` (hàm `get(k)` trả `0` nếu `k < 0`). Kiểm tra lại: chuyển tiếp ở `i = 3` với `d[3] == d[1]` thì `d[2]` bị trừ `d[0]` — đúng là tình huống TH2 (đoạn 4 đi cùng đường thẳng với đoạn 0).
- Vì phải sửa `d[i-1]`, ta làm việc trên **bản sao** `list(distance)`, tránh làm hỏng mảng của người gọi.

In [24]:
def is_self_crossing_spiral(distance):
    d = list(distance)                  # bản sao: ta sẽ sửa d[i-1] ở bước chuyển tiếp
    n = len(d)

    def get(k):                         # đoạn chưa tồn tại coi như dài 0
        return d[k] if k >= 0 else 0

    i = 2
    while i < n and d[i] > d[i - 2]:    # pha 1: xoắn ra
        i += 1
    if i >= n:
        return False                    # xoắn ra tới cuối: không bao giờ cắt

    if d[i] >= d[i - 2] - get(i - 4):   # bước chuyển tiếp: đoạn i vươn tới ngang tầm i-4
        d[i - 1] -= get(i - 3)          # bức tường cho đoạn i+1 gần lại

    i += 1
    while i < n:                        # pha 2: xoắn vào, mỗi đoạn phải ngắn hơn đoạn song song trước
        if d[i] >= d[i - 2]:
            return True
        i += 1
    return False


for d in ([2, 1, 1, 2], [1, 2, 3, 4], [1, 1, 1, 2, 1], [1, 1, 2, 2, 1, 1], [1, 1, 2, 2, 4, 1, 1]):
    print(d, "->", is_self_crossing_spiral(d))

[2, 1, 1, 2] -> True
[1, 2, 3, 4] -> False
[1, 1, 1, 2, 1] -> True
[1, 1, 2, 2, 1, 1] -> True
[1, 1, 2, 2, 4, 1, 1] -> False


### 8.3 Trace Cách 4

**`d = [2, 2, 3, 3, 2, 2]`** (đáp án `True`)

| Pha | `i` | So sánh | Hành động |
|---|---:|---|---|
| xoắn ra | 2 | `d[2]=3 > d[0]=2` ✓ | đi tiếp |
| xoắn ra | 3 | `d[3]=3 > d[1]=2` ✓ | đi tiếp |
| xoắn ra | 4 | `d[4]=2 > d[2]=3` ✗ | dừng: chuyển tiếp ở `i = 4` |
| chuyển tiếp | 4 | `d[4]=2 >= d[2]-d[0] = 1` ✓ | `d[3] = 3 - d[1] = 3 - 2 = 1` |
| xoắn vào | 5 | `d[5]=2 >= d[3]=1` ✓ | **trả `True`** |

**`d = [1, 1, 2, 2, 4, 1, 1]`** (đáp án `False`)

| Pha | `i` | So sánh | Hành động |
|---|---:|---|---|
| xoắn ra | 2, 3, 4 | `2>1`, `2>1`, `4>2` ✓ | đi tiếp |
| xoắn ra | 5 | `d[5]=1 > d[3]=2` ✗ | dừng: chuyển tiếp ở `i = 5` |
| chuyển tiếp | 5 | `d[5]=1 >= d[3]-d[1] = 1` ✓ | `d[4] = 4 - d[2] = 4 - 2 = 2` |
| xoắn vào | 6 | `d[6]=1 >= d[4]=2` ✗ | đi tiếp, hết mảng → **`False`** |

Ở ca thứ hai, việc thu ngắn `d[4]` từ `4` xuống `2` nghĩa là: đoạn 6 (đi xuống) chỉ được dài **dưới 2** mét, vì sau 2 mét nó sẽ chạm góc `(0, 1)` giữa đoạn 0 và đoạn 1. Đoạn 6 dài 1 nên an toàn.

In [25]:
def trace_spiral(distance):
    d = list(distance)
    n = len(d)
    get = lambda k: d[k] if k >= 0 else 0
    print("d =", d)
    i = 2
    while i < n and d[i] > d[i - 2]:
        print(f"  xoắn ra   i={i}: d[{i}]={d[i]} > d[{i-2}]={d[i-2]}")
        i += 1
    if i >= n:
        print("  => xoắn ra tới cuối, trả về False")
        return False
    need = d[i - 2] - get(i - 4)
    print(f"  chuyển tiếp i={i}: d[{i}]={d[i]} >= d[{i-2}]-d[{i-4}] = {need}? {yn(d[i] >= need)}", end="")
    if d[i] >= need:
        d[i - 1] -= get(i - 3)
        print(f" -> thu ngắn d[{i-1}] còn {d[i-1]}")
    else:
        print(" -> giữ nguyên")
    for k in range(i + 1, n):
        print(f"  xoắn vào  k={k}: d[{k}]={d[k]} >= d[{k-2}]={d[k-2]}? {yn(d[k] >= d[k-2])}")
        if d[k] >= d[k - 2]:
            print("  => cắt, trả về True")
            return True
    print("  => hết mảng, trả về False")
    return False


trace_spiral([2, 2, 3, 3, 2, 2])
print()
trace_spiral([1, 1, 2, 2, 4, 1, 1])

d = [2, 2, 3, 3, 2, 2]
  xoắn ra   i=2: d[2]=3 > d[0]=2
  xoắn ra   i=3: d[3]=3 > d[1]=2
  chuyển tiếp i=4: d[4]=2 >= d[2]-d[0] = 1? ✓ -> thu ngắn d[3] còn 1
  xoắn vào  k=5: d[5]=2 >= d[3]=1? ✓
  => cắt, trả về True

d = [1, 1, 2, 2, 4, 1, 1]
  xoắn ra   i=2: d[2]=2 > d[0]=1
  xoắn ra   i=3: d[3]=2 > d[1]=1
  xoắn ra   i=4: d[4]=4 > d[2]=2
  chuyển tiếp i=5: d[5]=1 >= d[3]-d[1] = 1? ✓ -> thu ngắn d[4] còn 2
  xoắn vào  k=6: d[6]=1 >= d[4]=2? ✗
  => hết mảng, trả về False


False

In [26]:
mismatch = sum(is_self_crossing_spiral(d) != is_self_crossing_simulation(d) for d in TESTS + extra)
print(f"Cách 4 so với trọng tài trên {len(TESTS) + len(extra)} test, số lần lệch: {mismatch}")
assert mismatch == 0

original = [1, 1, 2, 2, 1, 1]
is_self_crossing_spiral(original)
print("Mảng của người gọi sau khi gọi hàm:", original, "(không bị sửa nhờ list(distance))")

Cách 4 so với trọng tài trên 25144 test, số lần lệch: 0
Mảng của người gọi sau khi gọi hàm: [1, 1, 2, 2, 1, 1] (không bị sửa nhờ list(distance))


## 9. Vì sao đúng và độ phức tạp

### 9.1 Vì sao Cách 3 đúng?

Hai chiều cần chứng tỏ:

1. **Không báo nhầm** (mỗi lần trả `True` là cắt thật): mỗi điều kiện TH1/TH2/TH3 mô tả một **giao điểm cụ thể** giữa đoạn `i` và `i-3` / `i-4` / `i-5` — ta đã dựng tọa độ và suy ra chúng ở mục 5.3–5.5. Chúng đúng **bất kể** phần còn lại của đường đi ra sao.
2. **Không bỏ sót** (nếu có cắt thì sẽ trả `True`): xét đoạn `i` **đầu tiên** gây cắt. Trước nó đường đi chưa cắt, nên theo lập luận chữ U (mục 5.6), đoạn `i` chắc chắn đụng một trong `i-3`, `i-4`, `i-5` — tức một trong ba điều kiện đúng tại `i`. Vòng lặp duyệt `i` tăng dần nên sẽ trả `True` muộn nhất là ở bước này.

**Bất biến (invariant) của vòng lặp**: khi bắt đầu xét đoạn `i`, các đoạn `0..i-1` **chưa** cắt nhau (nếu đã cắt thì hàm đã `return True` trước đó). Chính bất biến này cho phép lập luận chữ U: "các bức tường `i-1`, `i-2`, `i-3` còn nguyên vẹn, chưa bị ai xuyên qua".

### 9.2 Vì sao Cách 4 đúng?

- Pha xoắn ra: mỗi đoạn nằm ngoài hình chữ nhật bao các đoạn cũ → không cắt.
- Bước chuyển tiếp: đoạn `i` (lần đầu `d[i] <= d[i-2]`) **tự nó không thể gây cắt**, vì đoạn `i-1` còn thuộc pha xoắn ra nên `d[i-1] > d[i-3]` — đoạn `i` xuất phát ngoài chữ U, mà cả ba trường hợp cắt đều cần `d[i-1] <= d[i-3]`. Nguy hiểm chỉ đến từ các đoạn **sau** nó; việc thu ngắn `d[i-1]` đặt lại đúng "bức tường" để pha xoắn vào phát hiện được cả kiểu đụng TH2/TH3.
- Pha xoắn vào: bất biến là "đoạn hiện tại nằm trong một hình chữ nhật được giới hạn bởi các đoạn gần nhất, và `d[k-2]` (sau khi điều chỉnh) là khoảng trống tối đa theo hướng của đoạn `k`". Vượt khoảng trống đó nghĩa là đâm vào tường.

### 9.3 Độ phức tạp (`n` = số đoạn, `S` = tổng độ dài)

| Cách | Thời gian | Bộ nhớ | Ghi chú |
|---|---|---|---|
| 1. Đi từng mét + `set` | `O(S)` trung bình, `S` tới `10^10` | `O(S)` | trực giác nhất, quá chậm |
| 2. Mô phỏng đoạn thẳng | `O(n²)` | `O(n)` | chắc chắn đúng → trọng tài |
| 3. Kiểm tra 3 trường hợp | `O(n)` | `O(1)` | **lời giải chính**, một lượt duyệt |
| 4. Xoắn ra / xoắn vào | `O(n)` | `O(n)` cho bản sao (hoặc `O(1)` nếu được sửa input) | hiểu được "hình dạng" đường đi |

Cách 3 chỉ dùng vài biến chỉ số, không có cấu trúc dữ liệu phụ → `O(1)` bộ nhớ thêm. Cell dưới đo thời gian thực tế: Cách 2 tăng gấp ~4 khi `n` gấp đôi (dấu hiệu `O(n²)`), còn Cách 3 và 4 chạy `n = 10^5` trong tích tắc.

In [27]:
def timeit(fn, d):
    t0 = time.perf_counter()
    fn(d)
    return time.perf_counter() - t0


for n in (500, 1000, 2000):
    d = list(range(1, n + 1))                                  # xoắn ra: trường hợp xấu nhất, phải xét hết
    print(f"Cách 2, n = {n:5d}: {timeit(is_self_crossing_simulation, d):.3f} s")

d = list(range(1, 10**5 + 1))
print(f"Cách 3, n = 100000: {timeit(is_self_crossing_cases, d):.3f} s")
print(f"Cách 4, n = 100000: {timeit(is_self_crossing_spiral, d):.3f} s")

Cách 2, n =   500: 0.040 s


Cách 2, n =  1000: 0.159 s


Cách 2, n =  2000: 0.658 s
Cách 3, n = 100000: 0.033 s
Cách 4, n = 100000: 0.006 s


## 10. Lỗi thường gặp

### Lỗi 1 — Dùng `>` / `<` thay cho `>=` / `<=` (quên rằng **chạm** cũng là cắt)

```python
if d[i] > d[i - 2] and d[i - 1] < d[i - 3]:   # SAI
```

Với `[1, 1, 1, 1]`, đoạn 3 kết thúc đúng tại điểm xuất phát; điều kiện "nghiêm ngặt" `1 > 1` sai → trả `False`. Đáp án đúng là `True`. **Sửa**: dùng `>=` / `<=` ở mọi chỗ (trừ `==` của TH2).

### Lỗi 2 — Chỉ cài TH1, quên TH2 / TH3

Nhiều người chỉ nghĩ tới hình "chữ thập" quen thuộc. Phản ví dụ:

- thiếu TH2: `[1, 1, 2, 1, 1]` (đoạn 4 chạm đầu đoạn 0 trên cùng đường thẳng);
- thiếu TH3: `[1, 1, 2, 2, 1, 1]` (đoạn 5 chạm đoạn 0 từ bên trong hộp).

### Lỗi 3 — Quên điều kiện `d[i-1] <= d[i-3]` trong TH3

Phản ví dụ `[1, 1, 2, 2, 3, 1]`: đoạn 4 vọt lên cao hơn trần, đoạn 5 chạy ra ngoài, không cắt gì — nhưng phiên bản thiếu điều kiện trả `True`.

Cell dưới cài các phiên bản sai và **tự động tìm phản ví dụ** bằng trọng tài. Đây là kỹ thuật cực kỳ hữu ích khi tự luyện: đừng chỉ chờ LeetCode báo *Wrong Answer*, hãy tự đối chiếu với brute force.

In [28]:
def wrong_strict(d):
    for i in range(3, len(d)):
        if d[i] > d[i - 2] and d[i - 1] < d[i - 3]:
            return True
        if i >= 4 and d[i - 1] == d[i - 3] and d[i] + d[i - 4] > d[i - 2]:
            return True
        if (i >= 5 and d[i - 2] > d[i - 4] and d[i] + d[i - 4] > d[i - 2]
                and d[i - 1] + d[i - 5] > d[i - 3] and d[i - 1] < d[i - 3]):
            return True
    return False


def wrong_only_case1(d):
    return any(d[i] >= d[i - 2] and d[i - 1] <= d[i - 3] for i in range(3, len(d)))


def wrong_no_cond4(d):
    for i in range(3, len(d)):
        if d[i] >= d[i - 2] and d[i - 1] <= d[i - 3]:
            return True
        if i >= 4 and d[i - 1] == d[i - 3] and d[i] + d[i - 4] >= d[i - 2]:
            return True
        if (i >= 5 and d[i - 2] >= d[i - 4] and d[i] + d[i - 4] >= d[i - 2]
                and d[i - 1] + d[i - 5] >= d[i - 3]):            # thiếu d[i-1] <= d[i-3]
            return True
    return False


def smallest_counterexample(fn):
    for d in TESTS:                                  # TESTS bắt đầu bằng các mảng ngắn nhất
        if fn(d) != is_self_crossing_simulation(d):
            return d, fn(d), is_self_crossing_simulation(d)
    return None


for fn in (wrong_strict, wrong_only_case1, wrong_no_cond4):
    print(f"{fn.__name__:17s}: phản ví dụ (input, trả về, đúng ra) = {smallest_counterexample(fn)}")

wrong_strict     : phản ví dụ (input, trả về, đúng ra) = ([1, 1, 1, 1], False, True)
wrong_only_case1 : phản ví dụ (input, trả về, đúng ra) = ([1, 1, 2, 1, 1], False, True)
wrong_no_cond4   : phản ví dụ (input, trả về, đúng ra) = ([1, 1, 2, 2, 3, 1], True, False)


### Lỗi 4 — Quên `i >= 4` / `i >= 5`: chỉ số âm âm thầm "quấn vòng"

```python
if d[i - 1] == d[i - 3] and d[i] + d[i - 4] >= d[i - 2]:   # SAI khi i = 3: d[-1] là phần tử CUỐI
```

Ở nhiều ngôn ngữ khác (Java, C++), `d[-1]` gây lỗi ngay nên dễ phát hiện. Python thì **không** — `d[-1]` hợp lệ và trả phần tử cuối mảng. Phản ví dụ `[1, 2, 1, 1]`: tại `i = 3`, `d[2] == d[0]` (`1 == 1`) và `d[3] + d[-1] = 1 + 1 = 2 >= d[1] = 2` → báo cắt. Nhưng đường đi thật `(0,0) → (0,1) → (-2,1) → (-2,0) → (-1,0)` không cắt gì.

**Sửa**: đặt `i >= 4` (và `i >= 5` cho TH3) ở **đầu** phép `and`, hoặc tách thành các vòng lặp riêng.

### Lỗi 5 — Cách 4: quên bước chuyển tiếp, hoặc sửa thẳng vào input

- Quên thu ngắn `d[i-1]` → sai với `[1, 1, 2, 2, 1, 1]` (mục 8.2). Phản ví dụ **nhỏ nhất** mà máy tìm được bên dưới còn ngắn hơn: `[1, 1, 2, 1, 1]` (chuyển tiếp ở `i = 3`, đoạn 4 chạm đoạn 0 kiểu TH2).
- Viết `d = distance` rồi `d[i-1] -= …` → **làm hỏng** mảng của người gọi (list là đối tượng *mutable*, gán `d = distance` chỉ tạo thêm một cái tên cho **cùng** một list). Trên LeetCode có thể vẫn AC, nhưng trong code thật đây là bug khó tìm. Dùng `d = list(distance)`.

In [29]:
def wrong_no_guard(d):
    for i in range(3, len(d)):
        if d[i] >= d[i - 2] and d[i - 1] <= d[i - 3]:
            return True
        if d[i - 1] == d[i - 3] and d[i] + d[i - 4] >= d[i - 2]:          # thiếu i >= 4
            return True
        if (d[i - 2] >= d[i - 4] and d[i] + d[i - 4] >= d[i - 2]           # thiếu i >= 5
                and d[i - 1] + d[i - 5] >= d[i - 3] and d[i - 1] <= d[i - 3]):
            return True
    return False


def wrong_spiral_no_transition(distance):
    d, n, i = list(distance), len(distance), 2
    while i < n and d[i] > d[i - 2]:
        i += 1
    for k in range(i + 1, n):          # bỏ qua bước chuyển tiếp
        if d[k] >= d[k - 2]:
            return True
    return False


def wrong_spiral_mutate(distance):
    d = distance                       # SAI: không phải bản sao
    n, i = len(d), 2
    while i < n and d[i] > d[i - 2]:
        i += 1
    if i < n and d[i] >= d[i - 2] - (d[i - 4] if i >= 4 else 0):
        d[i - 1] -= d[i - 3] if i >= 3 else 0
    return any(d[k] >= d[k - 2] for k in range(i + 1, n))


for fn in (wrong_no_guard, wrong_spiral_no_transition):
    print(f"{fn.__name__:27s}: phản ví dụ = {smallest_counterexample(fn)}")

arr = [1, 1, 2, 2, 1, 1]
print("wrong_spiral_mutate trả về", wrong_spiral_mutate(arr), "nhưng mảng đã bị đổi thành", arr)

wrong_no_guard             : phản ví dụ = ([1, 2, 1, 1], True, False)
wrong_spiral_no_transition : phản ví dụ = ([1, 1, 2, 1, 1], False, True)
wrong_spiral_mutate trả về True nhưng mảng đã bị đổi thành [1, 1, 2, 1, 1, 1]


## 11. Mẹo nhận dạng pattern và bài liên quan

### Dấu hiệu nhận ra kiểu bài này

- Đề mô tả **chuyển động trên lưới** theo các hướng cố định, xoay vòng → nghĩ ngay tới list `DIRS` và `i % 4`.
- Câu hỏi "có quay lại điểm cũ / có cắt đường cũ không" → lời giải **đầu tiên** nên nghĩ là `set` các điểm đã thăm (Cách 1). Nếu độ dài mỗi bước nhỏ, thế là đủ (ví dụ LeetCode 1496).
- Khi độ dài bước rất lớn → chuyển từ **điểm** sang **đoạn**; đoạn song song trục thì kiểm tra giao bằng **chồng khoảng** `max(lo) <= min(hi)` theo từng trục.
- Khi đề đòi `O(n)` / `O(1)` và hình có tính **lặp lại cục bộ** (mỗi bước chỉ phụ thuộc vài bước trước), hãy viết trọng tài rồi **làm thí nghiệm** để tìm xem thực sự cần nhìn lại bao xa — đúng như mục 5.1.
- Mọi ca "chạm" đều là ca biên: soát lại từng dấu `>=` / `<=`.

### Bài liên quan

| Bài | Độ khó | Liên hệ |
|---|---|---|
| 1496. Path Crossing | Easy | chuỗi `"NESW"` bước 1 mét: đúng Cách 1 với `set` |
| 874. Walking Robot Simulation | Medium | `DIRS` + xoay hướng + `set` vật cản |
| 2069. Walking Robot Simulation II | Medium | robot đi vòng quanh biên lưới, xoay hướng ngược chiều kim đồng hồ |
| 54. Spiral Matrix / 59. Spiral Matrix II | Medium | đi xoắn ốc vào trong, thu hẹp dần "tường" — đúng tinh thần Cách 4 |
| 885. Spiral Matrix III | Medium | đi xoắn ốc **ra** ngoài với độ dài đoạn tăng dần `1, 1, 2, 2, 3, 3, …` |
| 836. Rectangle Overlap | Easy | đúng phép kiểm tra chồng khoảng theo hai trục của `segments_intersect` |

Cell dưới giải nhanh **1496. Path Crossing** bằng đúng kỹ thuật Cách 1 (hai ví dụ của đề: `"NES"` → `false`, `"NESWW"` → `true`).

In [30]:
def is_path_crossing(path: str) -> bool:
    moves = {"N": (0, 1), "S": (0, -1), "E": (1, 0), "W": (-1, 0)}
    x, y = 0, 0
    visited = {(0, 0)}
    for ch in path:
        dx, dy = moves[ch]
        x, y = x + dx, y + dy
        if (x, y) in visited:
            return True
        visited.add((x, y))
    return False


print(is_path_crossing("NES"))     # False
print(is_path_crossing("NESWW"))   # True

False
True


## 12. Bài tập

Làm theo thứ tự, tự làm trước rồi mới xem **Gợi ý** (mục 13) và **Đáp án** (mục 14, cuối notebook).

### Bài 1 — Đoán output (không chạy code)

Mỗi input sau trả `True` hay `False`? Nếu `True`, đoạn nào cắt và thuộc trường hợp nào (TH1/TH2/TH3)?

| # | `distance` |
|---:|---|
| a | `[1, 2, 3, 4, 5]` |
| b | `[3, 3, 3, 3]` |
| c | `[2, 2, 4, 2, 2]` |
| d | `[5, 4, 3, 2, 1]` |
| e | `[1, 3, 2, 2, 1, 4]` |

### Bài 2 — Tính tay

Với `distance = [3, 3, 4, 2, 4]`:

1. Lập bảng tọa độ: điểm đầu, điểm cuối và hướng của từng đoạn.
2. Đoạn nào là đoạn đầu tiên gây cắt? Nó đụng đoạn nào, tại điểm nào? Đó là "cắt xuyên" hay "chạm"?
3. Trường hợp nào (TH1/TH2/TH3) của Cách 3 bắt được, ở `i` bằng mấy?

### Bài 3 — Điền TODO: ba trường hợp

Hoàn thiện `exercise_cases` bằng cách điền ba điều kiện. Nhớ: chạm cũng là cắt, và đừng để chỉ số âm lừa bạn.

In [31]:
def exercise_cases(distance):
    d = distance
    for i in range(3, len(d)):
        # TODO 1: TH1 — đoạn i cắt đoạn i-3 (2 điều kiện)
        # TODO 2: TH2 — đoạn i chạm đoạn i-4 (cần i >= 4, 2 điều kiện)
        # TODO 3: TH3 — đoạn i cắt đoạn i-5 (cần i >= 5, 4 điều kiện)
        pass
    return False


# Tự làm xong rồi mới mở test:
# assert exercise_cases([2, 1, 1, 2]) is True
# assert exercise_cases([1, 2, 3, 4]) is False
# assert exercise_cases([1, 1, 1, 2, 1]) is True
# assert exercise_cases([1, 1, 2, 2, 1, 1]) is True
# assert exercise_cases([1, 1, 2, 2, 3, 1]) is False
# assert exercise_cases([1, 2, 1, 1]) is False

### Bài 4 — Biến thể: đoạn đầu tiên gây cắt là đoạn nào?

Viết `first_crossing_index(distance)` trả về chỉ số `i` **nhỏ nhất** sao cho đoạn `i` cắt phần đường đã đi, hoặc `-1` nếu không cắt. Yêu cầu `O(n)` thời gian, `O(1)` bộ nhớ.

Gợi ý suy nghĩ: lời giải chính trả `True` ở bước `i` nào? Bước đó có đúng là đoạn cắt **đầu tiên** không — vì sao?

### Bài 5 — Biến thể: đi theo chiều kim đồng hồ

Nếu đổi luật thành Bắc → **Đông** → Nam → **Tây** (rẽ **phải** sau mỗi đoạn), đáp án có thay đổi không? Hãy viết `is_self_crossing_clockwise(distance)` bằng **mô phỏng đoạn thẳng** (dùng lại `segments_intersect`) với bộ hướng mới, chạy so với `is_self_crossing_cases` trên `TESTS`, rồi giải thích kết quả bằng một câu.

### Bài 6 — Mini challenge: tọa độ điểm cắt đầu tiên

Đề LeetCode có giải thích "tự cắt tại điểm `(0, 1)`", "tại điểm `(0, 0)`". Hãy viết `first_crossing_point(distance)` trả về **tọa độ** điểm đầu tiên (theo thứ tự đi) mà đường đi chạm lại phần cũ, hoặc `None`. Ràng buộc: độ dài mỗi đoạn có thể tới `10^5` nên **không được đi từng mét**; mục tiêu `O(n)`.

Kiểm tra: `[2, 1, 1, 2]` → `(0, 1)`, `[1, 1, 1, 2, 1]` → `(0, 0)`, `[3, 4, 5, 6, 3, 3]` → `(0, 1)`, `[1, 2, 3, 4]` → `None`.

In [32]:
def first_crossing_index(distance):
    # TODO: giống lời giải chính nhưng trả về i thay vì True, và -1 thay vì False
    raise NotImplementedError


def is_self_crossing_clockwise(distance):
    # TODO: DIRS_CW = [(0, 1), (1, 0), (0, -1), (-1, 0)]; dựng các đoạn rồi so từng cặp i, j <= i-3
    raise NotImplementedError


def first_crossing_point(distance):
    # TODO 1: i = first_crossing_index(distance); nếu -1 thì trả None
    # TODO 2: so đoạn i với mọi đoạn j <= i-3, lấy phần giao (một điểm hoặc một khúc thẳng)
    # TODO 3: trong các phần giao, chọn điểm gần điểm đầu của đoạn i nhất
    raise NotImplementedError


# Tự làm xong rồi mới mở test:
# assert first_crossing_index([1, 1, 2, 2, 1, 1]) == 5
# assert first_crossing_index([1, 2, 3, 4]) == -1
# assert all(is_self_crossing_clockwise(d) == is_self_crossing_cases(d) for d in TESTS)
# assert first_crossing_point([2, 1, 1, 2]) == (0, 1)
# assert first_crossing_point([1, 1, 1, 2, 1]) == (0, 0)

## 13. Gợi ý theo mức

**Bài 1**
- Mức 1: vẽ nhanh từng hình bằng tay hoặc nhẩm theo bảng `DIRS`.
- Mức 2: kiểm tra `d[i]` so với `d[i-2]`: dãy tăng mãi là xoắn ra, giảm mãi là xoắn vào.
- Mức 3: (c) chú ý `d[3] == d[1]`; (e) thử TH1 ở `i = 5`.

**Bài 2**
- Mức 1: dùng bảng hướng N, W, S, E và cộng dồn tọa độ.
- Mức 2: các điểm là `(0,0), (0,3), (-3,3), (-3,-1), (-1,-1), (-1,3)`.
- Mức 3: đoạn 4 đi lên dọc `x = -1`, kết thúc ở `y = 3` — đoạn 1 nằm trên `y = 3`.

**Bài 3**
- Mức 1: chép lại công thức ở mục 6.1, nhưng hãy thử tự suy ra từ hình trước.
- Mức 2: TH2 dùng `==` cho điều kiện "cùng đường thẳng"; mọi chỗ còn lại là `>=` / `<=`.
- Mức 3: `if i >= 4 and ...` — vế `i >= 4` phải đứng đầu.

**Bài 4**
- Mức 1: mỗi điều kiện TH1/2/3 tại `i` là một giao điểm **thật** của đoạn `i`.
- Mức 2: nên bước đầu tiên làm điều kiện đúng không thể đứng sau đoạn cắt đầu tiên, cũng không thể đứng trước.
- Mức 3: thay `return True` bằng `return i`, `return False` bằng `return -1`.

**Bài 5**
- Mức 1: lấy gương soi trục `y`: Tây ↔ Đông.
- Mức 2: viết `path_points_dirs(distance, dirs)` tổng quát rồi dựng đoạn như `segments_of`.
- Mức 3: phép đối xứng `x → -x` giữ nguyên việc hai đoạn có giao nhau hay không.

**Bài 6**
- Mức 1: tìm `i` bằng Bài 4 (`O(n)`), sau đó chỉ **một** đoạn `i` cần so với mọi đoạn cũ (`O(n)` nữa).
- Mức 2: phần giao của hai đoạn song song trục là "hộp" `[max(min x), min(max x)] × [max(min y), min(max y)]`.
- Mức 3: điểm gần điểm đầu `(sx, sy)` nhất trong hộp là `(clamp(sx), clamp(sy))` với `clamp(v) = min(max(v, lo), hi)`.

## 14. Đáp án tham khảo

### Đáp án Bài 1

| # | `distance` | Kết quả | Lý do |
|---:|---|---|---|
| a | `[1, 2, 3, 4, 5]` | `False` | xoắn ra: `d[i] > d[i-2]` với mọi `i` |
| b | `[3, 3, 3, 3]` | `True` | TH1 ở `i = 3` (`3 >= 3`, `3 <= 3`): đi một hình vuông, quay về đúng gốc |
| c | `[2, 2, 4, 2, 2]` | `True` | TH2 ở `i = 4`: `d[3] == d[1]` và `2 + 2 >= 4` → đoạn 4 chạm gốc `(0, 0)` |
| d | `[5, 4, 3, 2, 1]` | `False` | xoắn vào: `d[i] < d[i-2]` với mọi `i` |
| e | `[1, 3, 2, 2, 1, 4]` | `True` | TH1 ở `i = 5`: `d[5] = 4 >= d[3] = 2` và `d[4] = 1 <= d[2] = 2` → đoạn 5 cắt đoạn 2 |

### Đáp án Bài 2

| Đoạn | Hướng | Từ | Tới |
|---:|---|---|---|
| 0 | Bắc | `(0, 0)` | `(0, 3)` |
| 1 | Tây | `(0, 3)` | `(-3, 3)` |
| 2 | Nam | `(-3, 3)` | `(-3, -1)` |
| 3 | Đông | `(-3, -1)` | `(-1, -1)` |
| 4 | Bắc | `(-1, -1)` | `(-1, 3)` |

Đoạn 4 đi lên dọc `x = -1` và **kết thúc** ở `(-1, 3)` — điểm này nằm trên thân đoạn 1 (đoạn ngang `y = 3`, `x` từ `-3` tới `0`). Đây là **chạm** (hình chữ T), không phải cắt xuyên. Cách 3 bắt bằng **TH1 ở `i = 4`**: `d[4] = 4 >= d[2] = 4` (dấu `=`: vừa chạm) và `d[3] = 2 <= d[1] = 3`. (TH1 ở `i = 3` không thỏa vì `d[2] = 4 <= d[0] = 3` sai.)

### Đáp án Bài 3

Cell dưới vừa kiểm chứng đáp án Bài 1, Bài 2 bằng code, vừa chứa lời giải Bài 3 (đối chiếu với trọng tài trên toàn bộ `TESTS`).

In [33]:
answers_1 = {"a": ([1, 2, 3, 4, 5], False), "b": ([3, 3, 3, 3], True), "c": ([2, 2, 4, 2, 2], True),
             "d": ([5, 4, 3, 2, 1], False), "e": ([1, 3, 2, 2, 1, 4], True)}
for key, (d, expected) in answers_1.items():
    assert is_self_crossing_cases(d) == is_self_crossing_simulation(d) == expected
    print(key, d, "->", expected, "| first_crossing:", first_crossing(d))

d2 = [3, 3, 4, 2, 4]
assert path_points(d2) == [(0, 0), (0, 3), (-3, 3), (-3, -1), (-1, -1), (-1, 3)]
assert first_crossing(d2) == (4, [1]) and case1(d2, 4) and not case1(d2, 3)
draw_path(d2)


def exercise_cases_solution(distance):
    d = distance
    for i in range(3, len(d)):
        if d[i] >= d[i - 2] and d[i - 1] <= d[i - 3]:                                  # TH1
            return True
        if i >= 4 and d[i - 1] == d[i - 3] and d[i] + d[i - 4] >= d[i - 2]:            # TH2
            return True
        if (i >= 5 and d[i - 2] >= d[i - 4] and d[i] + d[i - 4] >= d[i - 2]            # TH3
                and d[i - 1] + d[i - 5] >= d[i - 3] and d[i - 1] <= d[i - 3]):
            return True
    return False


assert exercise_cases_solution([2, 1, 1, 2]) is True
assert exercise_cases_solution([1, 2, 3, 4]) is False
assert exercise_cases_solution([1, 1, 1, 2, 1]) is True
assert exercise_cases_solution([1, 1, 2, 2, 1, 1]) is True
assert exercise_cases_solution([1, 1, 2, 2, 3, 1]) is False
assert exercise_cases_solution([1, 2, 1, 1]) is False
assert all(exercise_cases_solution(d) == is_self_crossing_simulation(d) for d in TESTS)
print("Bài 3: PASS")

a [1, 2, 3, 4, 5] -> False | first_crossing: None
b [3, 3, 3, 3] -> True | first_crossing: (3, [0])
c [2, 2, 4, 2, 2] -> True | first_crossing: (4, [0])
d [5, 4, 3, 2, 1] -> False | first_crossing: None
e [1, 3, 2, 2, 1, 4] -> True | first_crossing: (5, [2])
+11111X11+
2     4  0
2     4  0
2     4  0
2     4  0
2     4  0
2     4  S
2     4
+33333+
Bài 3: PASS


### Đáp án Bài 4 và Bài 5

**Bài 4.** Mỗi điều kiện TH1/TH2/TH3 tại bước `i` là một giao điểm **thật** của đoạn `i` với đoạn cũ, nên bước đầu tiên làm điều kiện đúng **không thể đứng trước** đoạn cắt đầu tiên. Ngược lại, tại đoạn cắt đầu tiên, lập luận chữ U đảm bảo một điều kiện đúng, nên nó **không thể đứng sau**. Vậy chỉ cần đổi giá trị trả về. Ta đối chiếu với chỉ số `i` của `first_crossing` (trọng tài).

**Bài 5.** Đáp án **không đổi**. Đi theo chiều kim đồng hồ chính là ảnh **qua gương** (đối xứng qua trục `y`, `x → -x`) của đường đi ngược chiều kim đồng hồ với cùng `distance`. Phép đối xứng biến đoạn thành đoạn, giữ nguyên việc hai đoạn có điểm chung hay không, nên "tự cắt" được bảo toàn.

In [34]:
def first_crossing_index_solution(distance):
    d = distance
    for i in range(3, len(d)):
        if d[i] >= d[i - 2] and d[i - 1] <= d[i - 3]:
            return i
        if i >= 4 and d[i - 1] == d[i - 3] and d[i] + d[i - 4] >= d[i - 2]:
            return i
        if (i >= 5 and d[i - 2] >= d[i - 4] and d[i] + d[i - 4] >= d[i - 2]
                and d[i - 1] + d[i - 5] >= d[i - 3] and d[i - 1] <= d[i - 3]):
            return i
    return -1


def path_points_dirs(distance, dirs):
    x, y = 0, 0
    pts = [(0, 0)]
    for i, length in enumerate(distance):
        dx, dy = dirs[i % 4]
        x, y = x + dx * length, y + dy * length
        pts.append((x, y))
    return pts


def is_self_crossing_clockwise_solution(distance):
    DIRS_CW = [(0, 1), (1, 0), (0, -1), (-1, 0)]          # Bắc, Đông, Nam, Tây
    pts = path_points_dirs(distance, DIRS_CW)
    segs = [(pts[i], pts[i + 1]) for i in range(len(distance))]
    return any(segments_intersect(segs[i], segs[j]) for i in range(len(segs)) for j in range(i - 2))


assert first_crossing_index_solution([1, 1, 2, 2, 1, 1]) == 5
assert first_crossing_index_solution([1, 2, 3, 4]) == -1
for d in TESTS:
    fc = first_crossing(d)
    assert first_crossing_index_solution(d) == (fc[0] if fc else -1)
    assert is_self_crossing_clockwise_solution(d) == is_self_crossing_cases(d)
print("Bài 4: chỉ số đoạn cắt đầu tiên khớp trọng tài trên", len(TESTS), "test")
print("Bài 5: chiều kim đồng hồ cho cùng đáp án trên", len(TESTS), "test")

Bài 4: chỉ số đoạn cắt đầu tiên khớp trọng tài trên 24844 test
Bài 5: chiều kim đồng hồ cho cùng đáp án trên 24844 test


### Đáp án Bài 6

1. Tìm `i = first_crossing_index_solution(distance)` — `O(n)`. Nếu `-1` thì trả `None`.
2. So **duy nhất** đoạn `i` với mọi đoạn `j ≤ i-3` — `O(n)`. Phần giao của hai đoạn song song trục là "hộp" `[x_lo, x_hi] × [y_lo, y_hi]` (một điểm, hoặc một khúc thẳng nếu hai đoạn đè nhau).
3. Điểm của hộp gần điểm đầu `(sx, sy)` của đoạn `i` nhất là `(clamp(sx), clamp(sy))`. Vì hộp nằm trên chính đoạn `i`, khoảng cách tới điểm đầu đo bằng `|px - sx| + |py - sy|`. Chọn hộp cho khoảng cách nhỏ nhất.

Tổng cộng `O(n)` thời gian, `O(n)` bộ nhớ cho danh sách đoạn (có thể giảm về `O(1)` nếu tự tính lại các điểm, nhưng không cần thiết ở đây). Ta kiểm chứng với cách "đi từng mét" (điểm nguyên đầu tiên bị dẫm lại chính là điểm cắt đầu tiên) trên các test nhỏ.

In [35]:
def first_crossing_point_solution(distance):
    i = first_crossing_index_solution(distance)
    if i == -1:
        return None
    segs = segments_of(distance)
    (sx, sy), _ = segs[i]
    best = None
    for j in range(i - 2):
        (a, b), (c, e) = segs[i], segs[j]
        x_lo, x_hi = max(min(a[0], b[0]), min(c[0], e[0])), min(max(a[0], b[0]), max(c[0], e[0]))
        y_lo, y_hi = max(min(a[1], b[1]), min(c[1], e[1])), min(max(a[1], b[1]), max(c[1], e[1]))
        if x_lo <= x_hi and y_lo <= y_hi:
            px, py = min(max(sx, x_lo), x_hi), min(max(sy, y_lo), y_hi)
            dist = abs(px - sx) + abs(py - sy)
            if best is None or dist < best[0]:
                best = (dist, (px, py))
    return best[1]


def first_revisit_grid(distance):
    x, y = 0, 0
    visited = {(0, 0)}
    for i, length in enumerate(distance):
        dx, dy = DIRS[i % 4]
        for _ in range(length):
            x, y = x + dx, y + dy
            if (x, y) in visited:
                return (x, y)
            visited.add((x, y))
    return None


assert first_crossing_point_solution([2, 1, 1, 2]) == (0, 1)
assert first_crossing_point_solution([1, 1, 1, 2, 1]) == (0, 0)
assert first_crossing_point_solution([3, 4, 5, 6, 3, 3]) == (0, 1)
assert first_crossing_point_solution([1, 2, 3, 4]) is None
assert all(first_crossing_point_solution(d) == first_revisit_grid(d) for d in TESTS + extra)
big = [10**5, 10**5, 10**5, 10**5]
print("Bài 6: PASS; ví dụ lớn", big, "->", first_crossing_point_solution(big))

Bài 6: PASS; ví dụ lớn [100000, 100000, 100000, 100000] -> (0, 0)


## 15. Tổng kết

### Template ngắn dễ nhớ

```python
class Solution:
    def isSelfCrossing(self, distance: List[int]) -> bool:
        d = distance
        for i in range(3, len(d)):
            if d[i] >= d[i-2] and d[i-1] <= d[i-3]:                       # TH1: cắt i-3
                return True
            if i >= 4 and d[i-1] == d[i-3] and d[i] + d[i-4] >= d[i-2]:   # TH2: chạm i-4
                return True
            if (i >= 5 and d[i-2] >= d[i-4] and d[i] + d[i-4] >= d[i-2]   # TH3: cắt i-5
                    and d[i-1] + d[i-5] >= d[i-3] and d[i-1] <= d[i-3]):
                return True
        return False
```

Câu thần chú: **"ba – bốn – năm"**: cắt ngang `i-3`, đi thẳng chạm `i-4`, lọt khe cắt `i-5`. Và mọi trường hợp đều cần đoạn `i` xuất phát **trong chữ U**: `d[i-1] <= d[i-3]`.

### Checklist trước khi submit

- [ ] Mọi so sánh là `>=` / `<=` (chạm cũng là cắt), chỉ TH2 dùng `==` cho "cùng đường thẳng".
- [ ] Vòng lặp bắt đầu từ `i = 3`.
- [ ] TH2 có `i >= 4`, TH3 có `i >= 5`, đặt **đầu** phép `and` (tránh `d[-1]` âm thầm quấn vòng).
- [ ] TH3 đủ **bốn** điều kiện, đặc biệt `d[i-1] <= d[i-3]`.
- [ ] Đã chạy 3 ví dụ của đề và các ca `[1,1,1,1]`, `[1,1,2,1,1]`, `[1,1,2,2,1,1]`, `[1,1,2,2,3,1]`, `[1,2,1,1]`.
- [ ] Đã đối chiếu với một lời giải mô phỏng `O(n²)` trên dữ liệu ngẫu nhiên có `random.seed`.
- [ ] Nếu dùng Cách 4: có bước chuyển tiếp, và làm trên bản sao `list(distance)`.

### Bài học lớn

1. Bài hình học: **vẽ ra** trước khi nghĩ công thức.
2. Đi từ định nghĩa (đi từng mét → so từng cặp đoạn) để có **trọng tài** chắc chắn đúng.
3. Dùng trọng tài **làm thí nghiệm** để phát hiện cấu trúc ẩn (chỉ `i-3`, `i-4`, `i-5` quan trọng), rồi mới giải thích bằng hình và đặt tọa độ để ra điều kiện.
4. Kiểm chứng lời giải nhanh bằng đối chiếu chéo — và tự sinh phản ví dụ cho các phiên bản sai.